# **PRESENTACIÓN**

Este cuaderno convierte lo que publica `MODELADO_trabajo.ipynb` en una **página web**: `docs/index.html`. Es un solo archivo y se abre en cualquier navegador. GitHub Pages publica la carpeta `docs/` tal cual: basta con subir el cambio.

No ajusta ningún pronóstico: **lee** `outputs/`. Lo único que calcula son sumas y porcentajes. Antes de escribir la página comprueba que sus cifras coinciden con `metrics_main_by_season.csv` y `metrics_market_by_season.csv`.

**Cómo se usa.** Cada jornada, después de MODELADO: *Ejecutar todo*. Tarda unos segundos. Lo único que hay que tocar está en el apartado 0.2.

| Apartado | Qué hace |
|---|---|
| **0. Preparación** | dependencias y datos que se rellenan a mano (autor, enlaces) |
| **1. Lectura** | los ficheros de `outputs/` y de `data/` |
| **2. Cifras** | todo lo que enseña la página |
| **3. Diseño** | el estilo, los textos y los gráficos |
| **4. Página** | la monta, la comprueba, la escribe y la abre |

# **0. Preparación**

## 0.1 Dependencias e importes

In [1]:
# Este cuaderno no sale a la red.
!pip install -q pandas numpy pyarrow

In [2]:
# =============================================================================
#  CELDA · IMPORTES
# =============================================================================
import base64, glob, json, math, os, re
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import webbrowser

## 0.2 Lo que se rellena a mano

`DESDE` es la primera temporada que se enseña: todas las cifras, los gráficos y el buscador cuentan desde ella, como si se hubiera empezado a apostar entonces. El modelo se entrena igual con los partidos anteriores.

`LIMPIO_DESDE` es la primera jornada que **no** se había visto al elegir las variables del modelo. Las variables actuales se eligieron con datos hasta la jornada 13 de 2026/27, así que empieza en la 14. No sale en la página: el apartado 4 escribe debajo cómo va desde entonces. **Si se cambian las variables en MODELADO, hay que adelantar esta fecha.**

`CHAT` es la dirección del chat con el agente de datos, publicado en claude.ai. Con ella sale el apartado «Pregunta a los datos», con una vista previa y un botón que abre el chat; vacía, el apartado no sale. Para que lo puedan abrir los visitantes, el chat tiene que estar compartido con «cualquiera con el enlace» desde su menú Compartir.

`DIRECCION` es donde está publicada la página. Con ella, al compartir el enlace (LinkedIn, WhatsApp) sale la imagen `docs/portada.png`. Vacía, no sale imagen.

In [3]:
# =============================================================================
#  CELDA · CONFIGURACIÓN
# =============================================================================
AUTOR = "Manuel Fernández"
ENLACES = {                         # los que se dejan vacíos no salen
    "GitHub": "https://github.com/manuferrod",
    "LinkedIn": "https://www.linkedin.com/in/manuel-fernandez-rodriguez/",
}
DESDE = 2014                        # primera temporada que se enseña (2014 = 2014/15)
LIMPIO_DESDE = (2026, 14)           # (temporada, jornada)
DIRECCION = "https://manuferrod.github.io/pronostico-laliga/"
CHAT = "https://claude.ai/artifact/SFY1k5JMnt29qBwpwFw3Yw"   # chat con el agente de datos; vacío = no sale el apartado
ABRIR = True                        # abrir la página en el navegador al terminar

# **1. Lectura**

In [4]:
# =============================================================================
#  CELDA · LECTURA
# =============================================================================
ROOT = Path.cwd()
OUT, PROC, FEAT, RAW = ROOT / "outputs", ROOT / "data" / "02_processed", ROOT / "data" / "03_features", ROOT / "data" / "01_raw"
WEB = ROOT / "docs"
ESCUDOS = WEB / "escudos"
SIGNOS = ["H", "D", "A"]

_hay = [OUT / "metrics_main_by_season.csv", OUT / "metrics_market_by_season.csv", OUT / "matchlogs_2010.csv",
        FEAT / "df_final.parquet", PROC / "df_new_features.parquet"]
_no = [p.name for p in _hay if not p.exists()]
if _no:
    raise FileNotFoundError(f"faltan {_no}: ejecuta antes MODELADO_trabajo.ipynb")


def lee_logs(patron):
    fs = [f for f in sorted(glob.glob(str(OUT / patron))) if re.search(r"_\d{4}\.csv$", f)]
    d = pd.concat([pd.read_csv(f) for f in fs], ignore_index=True)
    d["Date"] = pd.to_datetime(d["Date"])
    return d.sort_values(["Date", "pred_key"], kind="mergesort").reset_index(drop=True)


ml = lee_logs("matchlogs_2*.csv")
mk = lee_logs("matchlogs_market_2*.csv")
met = pd.read_csv(OUT / "metrics_main_by_season.csv").set_index("Season")
metk = pd.read_csv(OUT / "metrics_market_by_season.csv").set_index("Season")
resumenes = sorted(OUT.glob("future_predictions_summary_*.json"), key=lambda p: p.stat().st_mtime)
MODELO = json.loads(resumenes[-1].read_text(encoding="utf-8"))["model"] if resumenes else None
df = pd.read_parquet(FEAT / "df_final.parquet").sort_values("Date", kind="mergesort").reset_index(drop=True)
bruto = pd.read_parquet(PROC / "df_new_features.parquet",
                        columns=["date", "home_canon", "away_canon", "FTR"])
marc = pd.read_parquet(PROC / "df_clean_vars.parquet", columns=["date", "home_canon", "away_canon", "FTHG", "FTAG"])

clave = ["Season", "Matchday", "Date", "HomeTeam_norm", "AwayTeam_norm"]
d = ml.merge(mk[clave + ["y_pred_market", "correct", "profit", "bet_placed"]], on=clave, suffixes=("", "_mk"), validate="one_to_one")
if len(d) != len(ml):
    raise ValueError("los matchlogs del modelo y los de Bet365 no tienen los mismos partidos")
marc["Date"] = pd.to_datetime(marc["date"])
marc["HomeTeam_norm"], marc["AwayTeam_norm"] = marc["home_canon"].str.lower(), marc["away_canon"].str.lower()
d = d.merge(marc[["Date", "HomeTeam_norm", "AwayTeam_norm", "FTHG", "FTAG"]], on=["Date", "HomeTeam_norm", "AwayTeam_norm"], how="left")
d = d[d["Season"] >= DESDE].reset_index(drop=True)          # el periodo que se enseña
jug = d[d["y_true"].isin(SIGNOS)].copy()

# **2. Cifras**

Los escudos se leen de `docs/escudos/<equipo>.png`. Si falta uno, el equipo sale con sus iniciales; para ponerlo basta con dejar ahí el archivo (el del Athletic, por ejemplo, es `ath_bilbao.png`).

In [5]:
# =============================================================================
#  CELDA · CIFRAS DE LA PÁGINA
# =============================================================================
NOMBRES = {
    "alaves": "Alavés", "almeria": "Almería", "ath bilbao": "Athletic", "ath madrid": "Atlético", "barcelona": "Barcelona",
    "betis": "Betis", "cadiz": "Cádiz", "celta": "Celta", "cordoba": "Córdoba", "eibar": "Eibar", "elche": "Elche",
    "espanyol": "Espanyol", "getafe": "Getafe", "girona": "Girona", "granada": "Granada", "hercules": "Hércules",
    "huesca": "Huesca", "la coruna": "Deportivo", "las palmas": "Las Palmas", "leganes": "Leganés", "levante": "Levante",
    "malaga": "Málaga", "mallorca": "Mallorca", "osasuna": "Osasuna", "oviedo": "Oviedo", "real madrid": "Real Madrid",
    "santander": "Racing", "sevilla": "Sevilla", "sociedad": "Real Sociedad", "sp gijon": "Sporting", "valencia": "Valencia",
    "valladolid": "Valladolid", "vallecano": "Rayo", "villarreal": "Villarreal", "zaragoza": "Zaragoza"}


def nombre(k):
    return NOMBRES.get(k, k.title())


def temporada(s):
    return f"{int(s)}/{(int(s) + 1) % 100:02d}"


def fiabilidad(p, acierto, cortes=np.arange(0, 1.0001, 0.1)):
    # una fila por partido y signo: probabilidad dada y si el signo salió
    q = np.clip(np.digitize(p, cortes) - 1, 0, len(cortes) - 2)
    t = pd.DataFrame({"q": q, "p": p, "y": acierto}).groupby("q").agg(p=("p", "mean"), y=("y", "mean"), n=("y", "size"))
    return [[round(float(r.p), 4), round(float(r.y), 4), int(r.n)] for r in t.itertuples() if r.n >= 60]


Y = np.array([[s == t for s in SIGNOS] for t in jug["y_true"]], float)
PM = jug[["proba_H", "proba_D", "proba_A"]].to_numpy()
PB = jug[["pH_mkt", "pD_mkt", "pA_mkt"]].to_numpy()
ap = jug[(jug["bet_placed"] == 1) & (jug["bet_placed_mk"] == 1)]
dif = 100 * (ap["profit"] - ap["profit_mk"])
ee = dif.std(ddof=1) / math.sqrt(len(dif))
dis = ap[ap["y_pred"] != ap["y_pred_market"]]


def bloque(x):
    return dict(n=int(len(x)), acierto=100 * x["correct"].mean(), acierto_mk=100 * x["correct_mk"].mean(),
                roi=100 * x["profit"].mean(), roi_mk=100 * x["profit_mk"].mean(),
                unidades=float(x["profit"].sum()), unidades_mk=float(x["profit_mk"].sum()),
                ok=int(x["correct"].sum()), ok_mk=int(x["correct_mk"].sum()))


porjor = ap.groupby(["Season", "Matchday"], sort=True).agg(n=("profit", "size"), m=("profit", "sum"), b=("profit_mk", "sum")).reset_index()
porjor["am"], porjor["ab"] = porjor["m"].cumsum(), porjor["b"].cumsum()

temps = []
for s, g in ap.groupby("Season"):
    v = 100 * (g["profit"] - g["profit_mk"])
    temps.append(dict(s=int(s), t=temporada(s), dis=int((g["y_pred"] != g["y_pred_market"]).sum()),
                      ventaja=float(v.mean()), ll=float(met.loc[s, "logloss"]), ll_mk=float(metk.loc[s, "logloss"]), **bloque(g)))

dis_signo = [[s, int(len(g)), int(g["correct"].sum()), int(g["correct_mk"].sum()), float(g["profit"].sum()), float(g["profit_mk"].sum())]
             for s, g in ((s, dis[dis["y_pred"] == s]) for s in SIGNOS)]
confusion = [[int(((jug["y_true"] == r) & (jug["y_pred"] == c)).sum()) for c in SIGNOS] for r in SIGNOS]

# cuatro variables de ejemplo, con el valor de cada equipo antes de su próximo partido
EJEMPLOS = [
    dict(c="perf_vs_market", dec=2, u="puntos por partido", q="¿Rinde más de lo que esperan las cuotas?", n="Puntos frente al mercado",
         d="Ganar al colista no vale lo mismo que ganar al líder. Cada resultado se compara con los puntos que la cuota daba por hechos, y los partidos recientes cuentan más.",
         f="puntos − (3 × prob. de ganar + prob. de empatar), media con semivida de 6 partidos", hoy="{a} es quien más supera lo esperado. {z}, quien más se queda corto."),
    dict(c="luck_last10", dec=2, u="goles por partido", q="¿Juega bien o tiene suerte?", n="Goles de más",
         d="El marcador a veces engaña. Esta variable separa los goles de las ocasiones: lo que un equipo marca y encaja frente a lo que debería por lo que genera y concede.",
         f="(goles − xG) + (xG en contra − goles en contra), media de 10 partidos", hoy="{a} saca más de lo que genera. {z}, menos."),
    dict(c="xgd_ewm", dec=2, u="goles esperados por partido", q="¿Quién domina de verdad?", n="Dominio con memoria",
         d="La distancia entre las ocasiones que un equipo crea y las que concede. Un partido pesa la mitad que otro jugado diez partidos después: manda el presente sin olvidar el pasado.",
         f="xG a favor − xG en contra, media con semivida de 10 partidos", hoy="{a} es quien más domina. {z}, quien más sufre."),
    dict(c="elo_change_last10", dec=0, u="puntos Elo", q="¿Va hacia arriba o hacia abajo?", n="Inercia",
         d="El rating Elo dice cómo de bueno es un equipo. Su cambio en los diez últimos partidos dice hacia dónde va, que no es lo mismo.",
         f="Elo de hoy − Elo de hace 10 partidos", hoy="{a} es quien más sube. {z}, quien más baja."),
]
_hayc = set(pq.ParquetFile(PROC / "df_new_features.parquet").schema_arrow.names)
EJEMPLOS = [e for e in EJEMPLOS if {f"home_{e['c']}", f"away_{e['c']}"} <= _hayc]
_nf = pd.read_parquet(PROC / "df_new_features.parquet", columns=["date", "season", "home_canon", "away_canon"] + [f"{s}_{e['c']}" for e in EJEMPLOS for s in ("home", "away")])
_nf = _nf[_nf["season"] == _nf["season"].max()]
_eq = pd.concat([_nf[["date", f"{s}_canon"] + [f"{s}_{e['c']}" for e in EJEMPLOS]].set_axis(["date", "k"] + [e["c"] for e in EJEMPLOS], axis=1) for s in ("home", "away")])
_eq["k"] = _eq["k"].str.lower()
_eq = _eq.sort_values("date", kind="mergesort").groupby("k").tail(1).set_index("k")      # el partido por jugar o, si no lo hay, el último
senales = []
for e in EJEMPLOS:
    v = _eq[e["c"]].dropna().sort_values(ascending=False)
    if len(v) >= 10:
        senales.append(dict(**{k: e[k] for k in ("q", "n", "d", "f", "u", "dec", "hoy")}, x=[[k, round(float(y), 3)] for k, y in v.items()]))

# la jornada por jugar, con el radar
RADAR = [("avg_xg_last7", "Goles esperados por partido", 2), ("avg_shotsontarget_last7", "Tiros a puerta por partido", 1),
         ("avg_corners_last7", "Córners por partido", 1), ("effectiveness", "Puntos por tiro a puerta", 2),
         ("form_points_6", "Puntos en los 6 últimos", 0), ("form_gd_6", "Diferencia de goles en los 6 últimos", 0),
         ("elo", "Rating Elo", 0), ("relative_perf", "Rendimiento frente a las cuotas", 2)]
proxima = []
fp = sorted(OUT.glob("future_predictions_2*.csv"))
if fp:
    f = pd.concat([pd.read_csv(x) for x in fp], ignore_index=True)
    f["Date"] = pd.to_datetime(f["Date"])
    f = f[f["pred_key"].isin(set(d.loc[~d["y_true"].isin(SIGNOS), "pred_key"]))].sort_values(["Date", "HomeTeam_norm"])
    rad = {}
    for x in (OUT / "radar_prematch").glob("radar_prematch_*.csv"):
        for r in pd.read_csv(x).to_dict("records"):
            rad[(r["HomeTeam_norm"], r["AwayTeam_norm"], int(r["Season"]))] = r
    for r in f.to_dict("records"):
        q = np.array([1 / r["B365H"], 1 / r["B365D"], 1 / r["B365A"]])
        q = q / q.sum()
        x = rad.get((r["HomeTeam_norm"], r["AwayTeam_norm"], int(r["Season"])))
        ficha = []
        if x:
            for c, rot, dec in RADAR:
                h, a = ("h_elo", "a_elo") if c == "elo" else (f"home_{c}", f"away_{c}")
                ficha.append([rot, dec, round(float(x[h]), 3), round(float(x[a]), 3), round(float(x[h + "_norm"]), 4), round(float(x[a + "_norm"]), 4)])
        proxima.append(dict(s=int(r["Season"]), j=int(r["Matchday"]), f=r["Date"].strftime("%Y-%m-%d"), l=r["HomeTeam_norm"], v=r["AwayTeam_norm"],
                            p=[round(float(r[c]), 4) for c in ("pH_pred", "pD_pred", "pA_pred")], m=[round(float(z), 4) for z in q],
                            o=[float(r["B365H"]), float(r["B365D"]), float(r["B365A"])], y=SIGNOS.index(r["y_pred"]), fav=int(q.argmax()), ficha=ficha))

equipos = sorted(set(jug["HomeTeam_norm"]) | set(jug["AwayTeam_norm"]) | {x["l"] for x in proxima} | {x["v"] for x in proxima})
ie = {k: i for i, k in enumerate(equipos)}
IX = {"H": 0, "D": 1, "A": 2}
partidos = [[int(r.Season), int(r.Matchday), r.Date.strftime("%Y%m%d"), ie[r.HomeTeam_norm], ie[r.AwayTeam_norm],
             -1 if pd.isna(r.FTHG) else int(r.FTHG), -1 if pd.isna(r.FTAG) else int(r.FTAG), IX[r.y_true], IX[r.y_pred], IX[r.y_pred_market],
             round(1000 * r.proba_H), round(1000 * r.proba_D), round(1000 * r.proba_A), r.B365H, r.B365D, r.B365A]
            for r in ap.itertuples()]


def png(k):
    p = ESCUDOS / f"{k.replace(' ', '_')}.png"
    return "data:image/png;base64," + base64.b64encode(p.read_bytes()).decode() if p.exists() else None


n_stats = sum(1 for _ in open(RAW / "sofascore" / "sofascore_stats.csv", encoding="utf-8")) - 1
cols = json.loads((FEAT / "df_final_columnas.json").read_text(encoding="utf-8"))
tot = bloque(ap)
DATOS = dict(
    generado=datetime.now().strftime("%Y-%m-%d"), hasta=ap["Date"].max().strftime("%Y-%m-%d"),
    total=dict(**tot, ventaja=float(dif.mean()), lo=float(dif.mean() - 1.96 * ee), hi=float(dif.mean() + 1.96 * ee),
               ll=float(-(Y * np.log(PM)).sum(1).mean()), ll_mk=float(-(Y * np.log(PB)).sum(1).mean()),
               brier=float(((PM - Y) ** 2).sum(1).mean()), brier_mk=float(((PB - Y) ** 2).sum(1).mean()),
               pct_dis=100 * len(dis) / len(ap), margen=100 * (float(ap["overround"].mean()) - 1)),
    discrepa=bloque(dis), dis_signo=dis_signo, temporadas=temps, confusion=confusion,
    curva=[[int(r.Season), int(r.Matchday), int(r.n), round(float(r.am), 2), round(float(r.ab), 2)] for r in porjor.itertuples()],
    fiab=dict(m=fiabilidad(PM.ravel(), Y.ravel()), b=fiabilidad(PB.ravel(), Y.ravel())),
    senales=senales, n_vars=len(MODELO["features"]) if MODELO else 0, proxima=proxima, partidos=partidos,
    modelo={k: MODELO[k] for k in ("n_seasons_window", "season_size", "recent_weight", "older_weight")} if MODELO else None,
    equipos=[dict(k=k, n=nombre(k), e=png(k)) for k in equipos],
    cadena=dict(partidos=int(bruto["FTR"].notna().sum()), desde=int(pd.to_datetime(bruto["date"]).min().year), stats=int(n_stats),
                tipos=int(pd.read_csv(RAW / "sofascore" / "sofascore_stats.csv", usecols=["event_id"])["event_id"].nunique()),
                cols_maestro=int(len(pq.ParquetFile(sorted(PROC.glob("fd_xg_elo_transfermarkt_wk_sofa_*.parquet"))[-1]).schema_arrow.names)),
                cols_feat=int(len(pq.ParquetFile(PROC / "df_new_features.parquet").schema_arrow.names)),
                variables=len(cols["variables"]), entrenables=int(len(df)), fechas=int(jug["Date"].nunique())))

# **3. Diseño**

Tres piezas: el estilo (CSS), los textos con huecos `{{así}}` que rellena el apartado 4 (HTML) y los gráficos y controles (JavaScript, sin librerías).

## 3.1 Estilo

In [6]:
# =============================================================================
#  CELDA · ESTILO
# =============================================================================
ESTILO = r'''/* El campo y el boleto. Lo que se mide va sobre césped segado (cada franja es una temporada);
   lo que se lee va sobre papel. Amarillo = el modelo, tiza discontinua = Bet365, en toda la página. */
:root {
  --cesped: #0e4a37; --cesped-2: #115541; --cesped-3: #0a3a2b; --tiza: #f3f8f4; --tiza-2: #b7d0c3; --tiza-3: rgba(243, 248, 244, .16);
  --modelo: #ffd23f; --casa: #c3d8cd;
  --papel: #f4f7f4; --blanco: #ffffff; --tinta: #10241b; --tinta-2: #4d6157; --tinta-3: #7a8b82; --linea: #d9e3dc; --linea-2: #c3d1c8;
  --verde: #0b6b4c; --verde-claro: #e3f1ea; --visita: #2c4f86; --mal: #a8323a; --mal-claro: #f7e6e7; --aviso: #7a5200; --aviso-claro: #fdf2d0;
  --titular: "Archivo", "Bahnschrift", "Segoe UI", system-ui, sans-serif;
  --texto: "Hanken Grotesk", "Segoe UI", system-ui, -apple-system, sans-serif;
  --radio: 10px;
}
* { box-sizing: border-box }
html { scroll-behavior: smooth; -webkit-text-size-adjust: 100%; color-scheme: light }
body { margin: 0; background: var(--papel); color: var(--tinta); font: 400 17px/1.6 var(--texto) }
[hidden] { display: none !important }
img { max-width: 100%; display: block }
a { color: inherit }
:focus-visible { outline: 3px solid var(--modelo); outline-offset: 2px; border-radius: 4px }
.papel :focus-visible { outline-color: var(--verde) }
.ancho { max-width: 1180px; margin-inline: auto; padding-left: max(20px, env(safe-area-inset-left, 0px)); padding-right: max(20px, env(safe-area-inset-right, 0px)) }
h1, h2, h3 { font-family: var(--titular); font-variation-settings: "wdth" 112; text-wrap: balance; margin: 0; letter-spacing: -.018em }
h1 { font-size: clamp(2.35rem, 5.4vw, 4.4rem); line-height: 1; font-weight: 800 }
h2 { font-size: clamp(1.7rem, 3.2vw, 2.6rem); line-height: 1.08; font-weight: 800 }
h3 { font-size: 1.14rem; line-height: 1.25; font-weight: 700; letter-spacing: -.005em }
p { margin: 0; max-width: 64ch }
.num { font-variant-numeric: tabular-nums lining-nums }
.chico { font-size: .875rem; line-height: 1.5; color: var(--tinta-2) }
.sr { position: absolute; width: 1px; height: 1px; overflow: hidden; clip-path: inset(50%) }

/* ── barra ── */
.barra { position: sticky; top: 0; z-index: 20; background: var(--cesped-3); color: var(--tiza) }
.barra .ancho { display: flex; align-items: center; gap: 24px; min-height: 56px }
.marca { display: flex; align-items: center; gap: 10px; font-family: var(--titular); font-variation-settings: "wdth" 108; font-weight: 700; text-decoration: none; white-space: nowrap }
.marca .signos { display: inline-grid; grid-auto-flow: column; gap: 2px; font-size: .72rem; font-weight: 800 }
.marca .signos i { font-style: normal; width: 20px; height: 22px; display: grid; place-items: center; border: 1px solid var(--tiza-2); border-radius: 3px }
.marca .signos i:first-child { background: var(--modelo); color: var(--cesped-3); border-color: var(--modelo) }
.barra nav { margin-left: auto; display: flex; gap: 4px; overflow-x: auto; scrollbar-width: none }
.barra nav a { text-decoration: none; font-size: .92rem; padding: 8px 12px; border-radius: 6px; white-space: nowrap; color: var(--tiza-2) }
.barra nav a:hover { color: var(--tiza); background: var(--tiza-3) }

/* ── portada ── */
.campo { background: var(--cesped); color: var(--tiza) }
.portada { padding-block: clamp(40px, 6vw, 76px) 0 }
.portada .cabeza { display: grid; grid-template-columns: minmax(0, 1.5fr) minmax(0, 1fr); gap: clamp(28px, 5vw, 72px); align-items: end }
.ante { font-size: .95rem; color: var(--tiza-2); margin-bottom: 18px }
.portada .entrada { margin-top: 22px; font-size: 1.14rem; line-height: 1.55; color: #dcebe3; max-width: 56ch }
.marcador { width: 100%; border-collapse: collapse; font-family: var(--titular) }
.marcador th { font-weight: 500; text-align: right; font-size: .9rem; color: var(--tiza-2); padding: 0 0 10px; font-family: var(--texto) }
.marcador th:first-child, .marcador td:first-child { text-align: left }
.marcador th:nth-child(n+2), .marcador td:nth-child(n+2) { padding-left: 26px }
.marcador td { border-top: 1px solid var(--tiza-3); padding-block: 12px; padding-right: 0; text-align: right; font-size: 1.5rem; font-weight: 600; font-variation-settings: "wdth" 104; font-variant-numeric: tabular-nums lining-nums; white-space: nowrap }
.marcador td:first-child { font: 400 1rem/1.3 var(--texto); color: #dcebe3; white-space: normal }
.marcador .m { color: var(--modelo); font-weight: 800 }
.marcador .b { color: var(--casa) }
.grafico-portada { margin-top: clamp(30px, 4vw, 52px) }
.pie-grafico { display: flex; flex-wrap: wrap; gap: 8px 28px; justify-content: space-between; padding-block: 14px 26px; color: var(--tiza-2); font-size: .92rem }
.pie-grafico p { max-width: 74ch }
.pie-grafico a { color: var(--tiza) }
.leyenda { display: flex; flex-wrap: wrap; gap: 6px 20px; font-size: .92rem; color: #dcebe3 }
.leyenda span { display: inline-flex; align-items: center; gap: 8px }
.leyenda i { width: 24px; height: 0; border-top: 3px solid var(--modelo) }
.leyenda i.casa { border-top: 2px dashed var(--casa) }
.leyenda i.ref { border-top: 1px solid var(--tiza-2) }
.grafico { position: relative; width: 100% }
.grafico svg { display: block; width: 100%; height: auto; overflow: visible; touch-action: pan-y }
.grafico text { font-family: var(--texto); font-variant-numeric: tabular-nums lining-nums }
.globo { position: absolute; pointer-events: none; z-index: 5; background: var(--blanco); color: var(--tinta); border-radius: 8px; padding: 9px 12px; font-size: .86rem; line-height: 1.45; box-shadow: 0 6px 24px rgba(4, 24, 16, .28); min-width: 150px; max-width: 260px; opacity: 0; transition: opacity .12s; left: 0; top: 0 }
.globo.ver { opacity: 1 }
.globo b { display: block; font-weight: 700; margin-bottom: 2px }
.globo div { display: flex; justify-content: space-between; gap: 18px; font-variant-numeric: tabular-nums }

/* ── secciones ── */
section, header.portada { scroll-margin-top: 56px }
section.papel { padding-block: clamp(56px, 8vw, 104px) }
section.papel.alt { background: var(--blanco) }
.titulo { display: grid; gap: 14px; margin-bottom: clamp(26px, 4vw, 44px) }
.titulo p { color: var(--tinta-2); font-size: 1.06rem }

/* ── boleto ── */
.boleto { background: var(--blanco); border: 1px solid var(--linea-2); border-radius: var(--radio); display: grid; grid-template-columns: 200px minmax(0, 1fr); box-shadow: 0 18px 50px -34px rgba(10, 58, 43, .55) }
.matriz { background: var(--cesped); color: var(--tiza); padding: 28px 24px; border-radius: var(--radio) 0 0 var(--radio); display: flex; flex-direction: column; gap: 18px; position: relative }
.matriz::after { content: ""; position: absolute; right: -1px; top: 10px; bottom: 10px; border-right: 3px dotted var(--blanco) }
.matriz .grande { font: 800 4.6rem/.85 var(--titular); font-variation-settings: "wdth" 100; color: var(--modelo) }
.matriz .lineas { flex: 1; min-height: 120px; display: grid; place-items: center; padding-block: 10px }
.matriz .lineas svg { height: 100%; max-height: 460px; width: 100%; stroke: var(--tiza); stroke-opacity: .3; fill: none }
.matriz dl { margin: 0; display: grid; gap: 12px; font-size: .9rem }
.matriz dt { color: var(--tiza-2) }
.matriz dd { margin: 0; font-weight: 600 }
.filas { min-width: 0 }
.fila-cab, .fila > summary { display: grid; grid-template-columns: minmax(0, 1fr) 372px 160px 22px; gap: 18px; align-items: center; padding-inline: 24px }
.fila-cab { padding-block: 14px; font-size: .84rem; color: var(--tinta-3); border-bottom: 1px solid var(--linea) }
.tres { display: grid; grid-template-columns: repeat(3, 1fr); gap: 8px; text-align: center }
.fila { border-bottom: 1px solid var(--linea) }
.fila:last-child { border-bottom: 0 }
.fila > summary { list-style: none; cursor: pointer; padding-block: 13px; min-height: 84px }
.fila > summary::-webkit-details-marker { display: none }
.fila > summary:hover { background: #f8fbf9 }
.duelo { display: grid; gap: 7px; min-width: 0 }
.duelo .eq { display: flex; align-items: center; gap: 12px; font-weight: 600; font-size: 1.06rem; line-height: 1.2; min-width: 0 }
.escudo { width: 28px; height: 28px; flex: none; object-fit: contain }
.escudo.sin { display: grid; place-items: center; border-radius: 50%; background: var(--verde-claro); color: var(--verde); font: 700 .62rem/1 var(--titular); letter-spacing: .02em }
.casilla { position: relative; border: 1px solid var(--linea-2); border-radius: 6px; padding: 8px 4px 10px; display: grid; gap: 1px; overflow: hidden; background: var(--blanco) }
.casilla .s { font: 600 .78rem/1 var(--titular); color: var(--tinta-3) }
.casilla .p { font: 700 1.2rem/1.15 var(--titular); font-variation-settings: "wdth" 100; font-variant-numeric: tabular-nums lining-nums }
.casilla .k { font-size: .76rem; line-height: 1.1; color: var(--tinta-3); font-variant-numeric: tabular-nums }
.casilla::after { content: ""; position: absolute; left: 0; bottom: 0; height: 3px; width: var(--w); background: var(--linea-2) }
.casilla.elegida { background: var(--cesped); border-color: var(--cesped); color: var(--tiza) }
.casilla.elegida .s { color: var(--modelo) } .casilla.elegida .k { color: var(--tiza-2) }
.casilla.elegida::after { background: var(--modelo) }
.etiqueta { display: inline-block; font-size: .8rem; line-height: 1.25; font-weight: 600; border-radius: 5px; padding: 4px 8px; background: var(--verde-claro); color: var(--verde) }
.etiqueta.ojo { background: var(--aviso-claro); color: var(--aviso) }
.flecha { width: 10px; height: 10px; border-right: 2px solid var(--tinta-3); border-bottom: 2px solid var(--tinta-3); transform: rotate(45deg); justify-self: center; transition: transform .15s }
.fila[open] .flecha { transform: rotate(225deg) }
.ficha { padding: 6px 24px 26px; display: grid; gap: 12px }
.ficha .quien { display: grid; grid-template-columns: 1fr 1fr; font-weight: 600; font-size: .92rem; padding-top: 10px; border-top: 1px dashed var(--linea-2) }
.ficha .quien span:last-child { text-align: right }
.met { display: grid; grid-template-columns: 64px 1fr 1fr 64px; gap: 4px 10px; align-items: center; font-size: .9rem }
.met .rot { grid-column: 1 / -1; text-align: center; font-size: .84rem; color: var(--tinta-2) }
.met .v { font-weight: 700; font-variant-numeric: tabular-nums } .met .v:last-child { text-align: right }
.met .b { height: 8px; background: var(--linea); border-radius: 4px; overflow: hidden; display: flex }
.met .b.i { justify-content: flex-end }
.met .b span { background: var(--tinta-3); border-radius: 4px } .met .b.gana span { background: var(--verde) }
.vacio { padding: 48px 28px; color: var(--tinta-2) }

/* ── paneles ── */
.panel { border-radius: var(--radio); padding: clamp(20px, 2.6vw, 32px); min-width: 0 }
.panel.campo { background: var(--cesped) }
.panel.hoja { background: var(--blanco); border: 1px solid var(--linea) }
.alt .panel.hoja { background: var(--papel); border-color: transparent }
.panel header { display: grid; gap: 8px; margin-bottom: 18px }
.panel.campo header p { color: var(--tiza-2); font-size: .95rem }
.panel.hoja header p { color: var(--tinta-2); font-size: .95rem }
.rejilla { display: grid; grid-template-columns: repeat(12, minmax(0, 1fr)); gap: 20px }
.c12 { grid-column: span 12 } .c7 { grid-column: span 7 } .c6 { grid-column: span 6 } .c5 { grid-column: span 5 } .c4 { grid-column: span 4 }
.duo { display: grid; grid-template-columns: 1fr 1fr; gap: 18px }
.duo div { display: grid; gap: 2px }
.duo .gr { font: 800 clamp(2rem, 3.6vw, 2.9rem)/1 var(--titular); font-variation-settings: "wdth" 102; font-variant-numeric: tabular-nums lining-nums }
.duo .gr.tenue { color: var(--tinta-3) }
.sep { border: 0; border-top: 1px solid var(--linea); margin: 22px 0 }
table.t { width: 100%; border-collapse: collapse; font-size: .92rem; font-variant-numeric: tabular-nums lining-nums }
table.t th { text-align: right; font-weight: 600; color: var(--tinta-2); padding: 8px 10px; border-bottom: 1px solid var(--linea-2); white-space: nowrap; vertical-align: bottom }
table.t td { text-align: right; padding: 8px 10px; border-bottom: 1px solid var(--linea); white-space: nowrap }
table.t th:first-child, table.t td:first-child { text-align: left; padding-left: 0 }
table.t td.mejor { color: var(--verde); font-weight: 700 }
.corre { overflow-x: auto }
details.tabla > summary { cursor: pointer; font-weight: 600; color: var(--verde); padding: 10px 0; width: max-content }
.trio { display: grid; gap: 18px }
.trio div { display: grid; grid-template-columns: 148px minmax(0, 1fr); gap: 16px; align-items: center; border-top: 1px solid var(--linea); padding-top: 18px }
.trio .gr { font: 800 2.3rem/1 var(--titular); font-variation-settings: "wdth" 102; font-variant-numeric: tabular-nums lining-nums }
.conf { display: grid; grid-template-columns: auto repeat(3, 1fr); gap: 4px; font-variant-numeric: tabular-nums }
.conf > * { padding: 8px 6px; text-align: center; font-size: .88rem }
.conf .cel { border-radius: 6px; padding: 14px 6px; font: 700 1.16rem/1.1 var(--titular) }
.conf .cel small { display: block; font: 400 .78rem/1.2 var(--texto); margin-top: 3px; opacity: .85 }
.conf .fi { text-align: left; align-self: center; color: var(--tinta-2) } .conf .co { color: var(--tinta-2); align-self: end }

/* ── explorador ── */
.mandos { display: flex; flex-wrap: wrap; gap: 14px; align-items: end; margin-bottom: 22px }
.mandos label { display: grid; gap: 5px; font-size: .88rem; color: var(--tinta-2) }
select, .boton { font: 600 1rem/1.2 var(--texto); color: var(--tinta); background: var(--blanco); border: 1px solid var(--linea-2); border-radius: 8px; padding: 11px 14px; min-height: 46px }
select { padding-right: 36px; min-width: 150px }
.boton { cursor: pointer } .boton:hover:not(:disabled) { background: var(--verde-claro) } .boton:disabled { opacity: .4; cursor: default }
.resumen { display: flex; flex-wrap: wrap; gap: 8px 30px; margin-bottom: 14px; font-size: .95rem; color: var(--tinta-2) }
.resumen b { color: var(--tinta); font-variant-numeric: tabular-nums }
.lista { display: grid }
.par { display: grid; grid-template-columns: minmax(0, 1fr) 62px minmax(0, 1fr) 92px 92px 78px; gap: 12px; align-items: center; padding: 11px 0; border-bottom: 1px solid var(--linea); font-size: .98rem }
.par.cab { font-size: .82rem; color: var(--tinta-3); padding-block: 0 8px }
.par .eq { display: flex; align-items: center; gap: 10px; min-width: 0; font-weight: 600 }
.par .eq span { overflow-wrap: anywhere; line-height: 1.2 }
.par .eq.l { justify-content: flex-end; text-align: right }
.par .res { text-align: center; font: 800 1.12rem/1 var(--titular); font-variant-numeric: tabular-nums }
.par .cel { text-align: center; font-variant-numeric: tabular-nums }
.pico { display: inline-flex; align-items: center; justify-content: center; gap: 6px; min-width: 66px; border-radius: 6px; padding: 5px 8px; font-weight: 700; background: var(--mal-claro); color: var(--mal) }
.pico .x { border-left: 1px solid currentColor; padding-left: 8px; margin-left: 2px; font-weight: 500 }
.pico.si { background: var(--verde-claro); color: var(--verde) }
.par .ben { text-align: right; font-weight: 600; font-variant-numeric: tabular-nums }
.par .escudo { width: 24px; height: 24px } .par .eq .escudo.sin { overflow-wrap: normal; font-size: .5rem; line-height: 1 }

/* ── proceso ── */
.pasos { display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); margin: 0 0 clamp(34px, 5vw, 56px); padding: 0; list-style: none; border-top: 3px solid var(--cesped) }
.pasos li { padding: 20px 22px 0 0; display: grid; gap: 8px; align-content: start; position: relative }
.pasos li::before { content: ""; position: absolute; top: -9px; left: 0; width: 15px; height: 15px; border-radius: 50%; background: var(--modelo); border: 3px solid var(--cesped) }
.pasos .dato { font: 800 1.7rem/1.1 var(--titular); font-variation-settings: "wdth" 102; font-variant-numeric: tabular-nums lining-nums }
.pasos p { font-size: .96rem; color: var(--tinta-2) }
.opt { margin: 0; padding: 0; list-style: none; counter-reset: p; display: grid; gap: 18px }
.opt li { counter-increment: p; display: grid; grid-template-columns: 40px minmax(0, 1fr); gap: 2px 16px; align-items: start }
.opt li::before { content: counter(p); grid-row: span 2; width: 40px; height: 44px; display: grid; place-items: center; border-radius: 6px; background: var(--cesped); color: var(--modelo); font: 800 1.15rem/1 var(--titular) }
.opt b { font-weight: 600; line-height: 1.35 } .opt p { color: var(--tinta-2); font-size: .96rem; line-height: 1.5 }
.embudo { display: flex; flex-direction: column } .cuadros { flex: 1; display: grid; grid-template-columns: repeat(18, minmax(0, 1fr)); gap: 5px; align-content: center }
.cuadros i { aspect-ratio: 1; border-radius: 3px; background: var(--tiza); opacity: .16 } .cuadros i.si { background: var(--modelo); opacity: 1 }
.senal-rej { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1.25fr); gap: 28px 48px; align-items: start }
.senal-tabs { display: grid; gap: 8px }
.senal-tabs button { font: 600 1.02rem/1.3 var(--texto); text-align: left; color: var(--tiza); background: transparent; border: 1px solid var(--tiza-3); border-radius: 8px; padding: 13px 16px; min-height: 48px; cursor: pointer }
.senal-tabs button:hover { border-color: var(--tiza-2) } .senal-tabs button[aria-selected="true"] { background: var(--modelo); border-color: var(--modelo); color: var(--cesped-3) }
.senal-txt { margin-top: 26px; display: grid; gap: 12px; color: var(--tiza-2) }
.senal-txt h4 { margin: 0; font: 800 1.5rem/1.15 var(--titular); font-variation-settings: "wdth" 106; color: var(--tiza) }
.senal-txt .formula { border-left: 3px solid var(--modelo); padding: 4px 0 4px 14px; color: var(--tiza); font-size: .96rem; line-height: 1.45 }
.senal-txt .hoy { color: var(--tiza) }
.notas { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 30px 48px; margin-top: 8px }
.notas div { display: grid; gap: 7px; align-content: start } .notas p { color: var(--tinta-2) }
.nota-fin { max-width: none; margin-top: clamp(28px, 4vw, 40px); padding-top: 18px; border-top: 1px solid var(--linea-2); color: var(--tinta-2); font-size: .92rem }
footer { background: var(--cesped-3); color: var(--tiza-2); padding-block: 44px; font-size: .94rem }
footer .ancho { display: grid; grid-template-columns: minmax(0, 1.4fr) minmax(0, 1fr); gap: 30px 60px }
footer b { color: var(--tiza) } footer a { color: var(--tiza) } footer p + p { margin-top: 10px }
.enlaces { display: flex; flex-wrap: wrap; gap: 8px 18px; margin-top: 10px }

@media (max-width: 1000px) {
  .portada .cabeza { grid-template-columns: 1fr; align-items: start }
  .c7, .c6, .c5, .c4 { grid-column: span 12 }
  .boleto { grid-template-columns: 1fr }
  .matriz { border-radius: var(--radio) var(--radio) 0 0; flex-direction: row; align-items: center; flex-wrap: wrap; padding: 20px 24px }
  .matriz::after { right: 10px; left: 10px; top: auto; bottom: -2px; border-right: 0; border-bottom: 3px dotted var(--blanco) }
  .matriz .grande { font-size: 3.2rem } .matriz .lineas { display: none } .matriz dl { margin: 0 0 0 auto; grid-auto-flow: column; gap: 26px }
  .senal-rej { grid-template-columns: 1fr } .senal-tabs { grid-template-columns: 1fr 1fr } .cuadros { grid-template-columns: repeat(28, minmax(0, 1fr)) }
  .pasos { grid-template-columns: 1fr 1fr; row-gap: 34px } .pasos li:nth-child(n+3) { border-top: 3px solid var(--cesped) }
  .fila-cab, .fila > summary { grid-template-columns: minmax(0, 1fr) 300px 22px } .fila-cab > :nth-child(3), .fila .et { display: none }
  .duelo .et2 { display: block !important }
}
@media (max-width: 680px) {
  body { font-size: 16px }
  .barra .ancho { flex-wrap: wrap; gap: 0 } .barra nav { margin: 0 -12px; width: calc(100% + 24px); padding-bottom: 6px }
  .marca { padding-top: 10px }
  .fila-cab { display: none }
  .fila > summary { grid-template-columns: minmax(0, 1fr) 22px; padding-inline: 16px; gap: 12px } .fila > summary .tres { grid-column: 1 / -1; grid-row: 2 }
  .ficha { padding-inline: 16px } .matriz { padding-inline: 16px } .matriz dl { margin: 0; grid-auto-flow: row; gap: 4px } .matriz dl div { display: flex; gap: 8px }
  .par, .par.cab { grid-template-columns: minmax(0, 1fr) 46px minmax(0, 1fr); gap: 6px 8px } .par.cab { display: none }
  .par .cel, .par .ben { grid-row: 2; font-size: .9rem } .par .cel:nth-of-type(1) { text-align: left } .par .cel.m { grid-column: 1; text-align: left } .par .cel.c { grid-column: 2 / 3; white-space: nowrap } .par .ben { grid-column: 3 }
  .par .escudo { width: 20px; height: 20px }
  .conf { gap: 3px } .conf > * { padding: 6px 3px; font-size: .8rem } .conf .cel { padding: 11px 2px; font-size: 1rem }
  table.t { font-size: .86rem } table.t th, table.t td { padding-inline: 6px } table.t th:last-child, table.t td:last-child { padding-right: 0 }
  .pasos { grid-template-columns: 1fr; border-top: 0; border-left: 3px solid var(--cesped); margin-left: 8px; row-gap: 30px } .pasos li, .pasos li:nth-child(n+3) { padding: 0 0 0 24px; border-top: 0 } .pasos li::before { top: 3px; left: -12px }
  .notas { grid-template-columns: 1fr } footer .ancho { grid-template-columns: 1fr }
  .senal-tabs { grid-template-columns: 1fr } .cuadros { grid-template-columns: repeat(18, minmax(0, 1fr)); gap: 4px } .marcador td { font-size: 1.25rem }
  .met { grid-template-columns: 52px 1fr 1fr 52px }
}
/* ── pregunta a los datos: el chat, como producto aparte ── */
section.producto { padding-block: clamp(56px, 8vw, 104px) }
.producto-rej { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1.05fr); gap: clamp(28px, 5vw, 64px); align-items: center }
.producto-txt .entrada { margin-top: 18px; font-size: 1.1rem; line-height: 1.55; color: #dcebe3; max-width: 52ch }
.ventajas { list-style: none; padding: 0; margin: 24px 0 28px; display: grid; gap: 12px; max-width: 52ch }
.ventajas li { position: relative; padding-left: 24px; color: var(--tiza-2) }
.ventajas li::before { content: ""; position: absolute; left: 0; top: .55em; width: 10px; height: 10px; border-radius: 2px; background: var(--modelo) }
.ventajas b { color: var(--tiza) }
.cta { display: inline-flex; align-items: center; gap: 10px; background: var(--modelo); color: var(--cesped-3); font: 800 1.06rem var(--titular); font-variation-settings: "wdth" 108; text-decoration: none; padding: 14px 26px; border-radius: 10px }
.cta:hover { background: #ffe07a }
.cta-nota { margin-top: 12px; font-size: .88rem; color: var(--tiza-2) }
.ventana { margin: 0; background: var(--papel); color: var(--tinta); border-radius: 16px; padding: clamp(16px, 2vw, 22px); display: grid; gap: 12px; box-shadow: 0 18px 44px rgba(0, 0, 0, .28); min-width: 0 }
.ventana figcaption { font-size: .74rem; text-transform: uppercase; letter-spacing: .08em; color: var(--tinta-3); font-weight: 600 }
.burbuja { padding: 10px 14px; max-width: 92%; font-size: .98rem }
.burbuja.yo { justify-self: end; background: var(--cesped); color: var(--tiza); border-radius: 14px 14px 4px 14px }
.burbuja.agente { background: var(--blanco); border: 1px solid var(--linea); border-radius: 4px 14px 14px 14px; display: grid; gap: 10px; max-width: 100%; min-width: 0 }
.burbuja.agente p { max-width: none }
.paso-chat { justify-self: start; font-size: .72rem; font-weight: 700; text-transform: uppercase; letter-spacing: .06em; color: var(--verde); background: var(--verde-claro); border-radius: 6px; padding: 3px 8px }
.mini-campo { background: var(--cesped); border-radius: 8px; padding: 12px 14px 8px; display: grid; gap: 8px }
.grafico.mini .cero { stroke: var(--tiza-3); stroke-width: 1 }
.grafico.mini .l-modelo { fill: none; stroke: var(--modelo); stroke-width: 3; stroke-linejoin: round }
.grafico.mini .l-casa { fill: none; stroke: var(--casa); stroke-width: 2; stroke-dasharray: 6 5 }
.grafico.mini .v-modelo { fill: var(--modelo); font-weight: 700; font-size: 12px }
.grafico.mini .v-casa { fill: var(--casa); font-weight: 600; font-size: 12px }
.grafico.mini .eje { fill: var(--tiza-2); font-size: 10.5px }
@media (max-width: 1000px) { .producto-rej { grid-template-columns: 1fr } }
@media (prefers-reduced-motion: reduce) { html { scroll-behavior: auto } * { transition: none !important } }
@media print { .barra, .mandos { display: none } }
'''

## 3.2 Textos y estructura

In [7]:
# =============================================================================
#  CELDA · TEXTOS Y ESTRUCTURA
# =============================================================================
CUERPO = r'''<div class="barra"><div class="ancho">
  <a class="marca" href="#inicio"><span class="signos" aria-hidden="true"><i>1</i><i>X</i><i>2</i></span>Pronóstico LaLiga</a>
  <nav aria-label="Secciones"><a href="#jornada">Jornada</a><a href="#resultados">Resultados</a><a href="#explorador">Partidos</a>{{nav_chat}}<a href="#proceso">Cómo funciona</a><a href="#reglas">Reglas</a></nav>
</div></div>

<main>
<header class="campo portada" id="inicio"><div class="ancho">
  <div class="cabeza">
    <div>
      <p class="ante">Proyecto de analítica deportiva · LaLiga {{primera}} a {{ultima}}</p>
      <h1>¿Se puede pronosticar LaLiga mejor que Bet365?</h1>
      <p class="entrada">Un modelo pronostica local, empate o visitante en cada partido de Primera. Lo comparo, uno a uno, con el favorito de la casa de apuestas en {{n}} partidos.</p>
    </div>
    <table class="marcador">
      <thead><tr><th><span class="sr">Medida</span></th><th>Modelo</th><th>Favorito<br>de Bet365</th></tr></thead>
      <tbody>
        <tr><td>Partidos acertados</td><td class="m">{{acierto}}</td><td class="b">{{acierto_mk}}</td></tr>
        <tr><td>Rentabilidad por apuesta</td><td class="m">{{roi}}</td><td class="b">{{roi_mk}}</td></tr>
        <tr><td>Unidades tras {{temporadas}} temporadas</td><td class="m">{{unidades}}</td><td class="b">{{unidades_mk}}</td></tr>
      </tbody>
    </table>
  </div>
  <div class="grafico-portada">
    <div class="leyenda" style="margin-bottom:12px"><span><i></i>Modelo</span><span><i class="casa"></i>Bet365, siempre al favorito</span></div>
    <div class="grafico" id="g-curva" role="img" aria-label="Beneficio acumulado apostando una unidad a cada partido: el modelo acaba en {{unidades}} unidades y el favorito de Bet365 en {{unidades_mk}}."></div>
    <div class="pie-grafico"><p>Beneficio acumulado apostando 1 unidad a cada partido desde {{primera}}, con la cuota de apertura de Bet365. Cada franja es una temporada y cada partido se pronostica solo con datos anteriores a su fecha. <a href="#reglas">Ver las reglas</a></p></div>
  </div>
</div></header>

<section class="papel" id="jornada"><div class="ancho">
  <div class="titulo"><h2>{{titulo_jornada}}</h2><p>{{texto_jornada}}</p></div>
  <div id="boleto"></div>
</div></section>

<section class="papel alt" id="resultados"><div class="ancho">
  <div class="titulo"><h2>{{temporadas_letra}} temporadas frente al mercado</h2>
  <p>En {{ganadas}} de {{temporadas}} temporadas el modelo rinde más que apostar siempre al favorito. La ventaja media es de {{ventaja}} puntos de rentabilidad, con un intervalo del 95 % entre {{lo}} y {{hi}}.</p></div>
  <div class="rejilla">
    <div class="panel campo c12"><header><h3>Ventaja sobre Bet365 en cada temporada</h3><p>Puntos de rentabilidad: la del modelo menos la de apostar al favorito. {{ultima}} va por la jornada {{jornada_actual}}.</p></header>
      <div class="grafico" id="g-ventaja" role="img" aria-label="Ventaja de rentabilidad del modelo sobre Bet365 por temporada. Los valores están en la tabla de debajo."></div></div>
    <div class="c12"><details class="tabla"><summary>Ver la tabla por temporada</summary><div class="corre" id="t-temporadas"></div></details></div>

    <div class="panel hoja c5"><header><h3>Dónde gana el modelo</h3><p>Coincide con el favorito casi siempre. Solo se aparta en el {{pct_dis}} de los partidos ({{n_dis}}), y ahí acierta más.</p></header>
      <div class="duo"><div><span class="gr">{{dis_ac}}</span><span class="chico">acierta el modelo</span></div><div><span class="gr tenue">{{dis_ac_mk}}</span><span class="chico">acierta el favorito</span></div></div>
      <hr class="sep">
      <div class="duo"><div><span class="gr">{{dis_u}}</span><span class="chico">unidades del modelo</span></div><div><span class="gr tenue">{{dis_u_mk}}</span><span class="chico">unidades del favorito</span></div></div>
      <hr class="sep">
      <div class="corre" id="t-dis"></div>
    </div>
    <div class="panel campo c7"><header><h3>¿Se cumplen las probabilidades?</h3><p>De los pronósticos al 60 %, ¿cuántos salen? Cuanto más cerca de la diagonal, más fiable es la probabilidad. Pérdida logarítmica: {{ll}} el modelo y {{ll_mk}} Bet365 (menos es mejor).</p></header>
      <div class="leyenda" style="margin-bottom:10px"><span><i></i>Modelo</span><span><i class="casa"></i>Bet365</span><span><i class="ref"></i>Calibración perfecta</span></div>
      <div class="grafico" id="g-fiab" role="img" aria-label="Calibración: probabilidad pronosticada frente a frecuencia real, para el modelo y para Bet365."></div></div>

    <div class="panel hoja c7"><header><h3>Qué pronostica y qué pasa</h3><p>Partidos según el signo pronosticado y el resultado. El porcentaje es sobre cada resultado; los aciertos van con borde amarillo.</p></header>
      <div class="corre"><div class="conf" id="confusion"></div></div></div>
    <div class="panel hoja c5"><header><h3>El empate, el signo difícil</h3><p>Es frecuente, pero casi nunca es el más probable de los tres.</p></header>
      <div class="trio"><div><span class="gr">{{pct_empates}}</span><span class="chico">de los partidos acaba en empate</span></div><div><span class="gr">{{pct_pred_empate}}</span><span class="chico">de las veces lo pronostica el modelo ({{n_pred_empate}} partidos)</span></div><div><span class="gr">{{ac_empate}}</span><span class="chico">de acierto cuando lo pronostica</span></div></div></div>
  </div>
</div></section>

<section class="papel" id="explorador"><div class="ancho">
  <div class="titulo"><h2>Todos los partidos</h2><p>Elige temporada y jornada para ver cada pronóstico con su resultado.</p></div>
  <div class="mandos">
    <label for="sel-t">Temporada<select id="sel-t"></select></label>
    <label for="sel-j">Jornada<select id="sel-j"></select></label>
    <button class="boton" id="ant" type="button">Jornada anterior</button><button class="boton" id="sig" type="button">Jornada siguiente</button>
  </div>
  <div class="rejilla">
    <div class="panel campo c12"><header><h3 id="exp-tit"></h3><p id="exp-sub"></p></header>
      <div class="leyenda" style="margin-bottom:10px"><span><i></i>Modelo</span><span><i class="casa"></i>Bet365, siempre al favorito</span></div>
      <div class="grafico" id="g-temp" role="img" aria-label="Beneficio acumulado en la temporada elegida."></div></div>
    <div class="panel hoja c12"><header><h3 id="jor-tit"></h3></header><div class="resumen" id="jor-res"></div><div class="lista" id="jor-lista"></div></div>
  </div>
</div></section>
{{seccion_chat}}
<section class="papel alt" id="proceso"><div class="ancho">
  <div class="titulo"><h2>Cómo funciona</h2><p>Cuatro pasos en Python, del dato público al pronóstico. Se ejecutan en orden cada jornada y cada uno comprueba sus datos antes de pasar al siguiente.</p></div>
  <ol class="pasos">
    <li><h3>Extracción</h3><span class="dato">{{c_partidos}} partidos</span><p>Cinco fuentes públicas desde {{c_desde}}: football-data (resultados y cuotas), Understat y SofaScore (goles esperados y estadística), ClubElo (nivel de cada club) y Transfermarkt (valor de plantilla).</p></li>
    <li><h3>Limpieza y variables</h3><span class="dato">{{c_stats}} datos</span><p>La estadística de {{c_tipos}} partidos se convierte en rachas, promedios y tendencias. Ninguna usa lo que pasa en el propio partido.</p></li>
    <li><h3>Preprocesado</h3><span class="dato">{{c_vars}} variables</span><p>Se comparan los dos equipos, se tratan los huecos y queda una tabla de {{c_entren}} partidos lista para entrenar.</p></li>
    <li><h3>Modelado</h3><span class="dato">{{n_vars}} variables finales</span><p>Regresión logística multinomial, reentrenada antes de cada fecha de partidos. Es un modelo sencillo que se puede explicar.</p></li>
  </ol>
  <div class="rejilla">
    <div class="panel campo c12"><header><h3>Cada fecha se pronostica con su pasado</h3><p>Para pronosticar un día se entrena con los {{ventana}} partidos anteriores (cuatro temporadas). Los {{recientes}} últimos pesan el {{peso_letra}}. Al día siguiente la ventana avanza y se repite.</p></header>
      <div class="grafico" id="g-wf" role="img" aria-label="Esquema de validación: una ventana de {{ventana}} partidos avanza en el tiempo y pronostica la fecha siguiente."></div></div>
    <div class="panel campo c12 senal"><header><h3>De la estadística a la señal</h3><p>Un modelo no entiende de fútbol: hay que traducírselo. Cada variable nace de una pregunta. Estas son cuatro de las {{c_vars}} creadas, con los datos {{momento}}.</p></header>
      <div class="senal-rej">
        <div><div class="senal-tabs" id="senal-tabs" role="tablist" aria-label="Variables de ejemplo"></div><div id="senal-txt" class="senal-txt" role="tabpanel" aria-live="polite"></div></div>
        <div class="grafico" id="g-senal" role="img" aria-label="Valor de la variable elegida en cada equipo de la temporada."></div>
      </div></div>
    <div class="panel campo c5 embudo"><header><h3>De {{c_vars}} candidatas a {{n_vars}}</h3><p>Cada cuadro es una variable creada. En amarillo, las que se quedan.</p></header><div class="cuadros" id="cuadros" aria-hidden="true"></div></div>
    <div class="panel hoja c7"><header><h3>Cómo se optimiza</h3><p>Paso a paso, y siempre con el examen completo.</p></header>
      <ol class="opt">
        <li><b>Añadir o quitar una</b><p>Se parte de un conjunto pequeño y se prueba cada candidata, de una en una.</p></li>
        <li><b>Repetir el examen</b><p>Con cada prueba el modelo se vuelve a entrenar y a pronosticar fecha a fecha: {{c_fechas}} entrenamientos en el periodo que se ve aquí.</p></li>
        <li><b>Dos notas a la vez</b><p>Se mira cuánto acierta y cuánto rinde. La diferencia con el conjunto anterior se mide partido a partido.</p></li>
        <li><b>Quedarse con poco</b><p>Si una variable no aporta, sale. Un modelo pequeño es más estable y se puede explicar.</p></li>
      </ol></div>
  </div>
</div></section>

<section class="papel" id="reglas"><div class="ancho">
  <div class="titulo"><h2>Las reglas</h2><p>Un pronóstico solo vale si la prueba es justa. Así se mide este.</p></div>
  <div class="notas">
    <div><h3>Nunca ve el futuro</h3><p>Para pronosticar un partido, el modelo solo se entrena con los anteriores. Ninguna variable usa lo que pasa en el propio partido.</p></div>
    <div><h3>Mismos partidos, mismas cuotas</h3><p>El modelo y el favorito de Bet365 se miden en los mismos {{n}} partidos y a la misma cuota: la de apertura, con un margen medio del {{margen}} a favor de la casa.</p></div>
    <div><h3>Se apuesta a todo</h3><p>Una unidad en cada partido. No se eligen los fáciles ni se cambia la cantidad según la confianza.</p></div>
    <div><h3>Todo se comprueba</h3><p>Cada paso revisa sus datos antes de dar paso al siguiente, y esta página no se publica si sus cifras no coinciden con las del modelo.</p></div>
  </div>
  <p class="nota-fin">Es una simulación con datos históricos. Las variables se seleccionaron con estas mismas temporadas.</p>
</div></section>
</main>

<footer><div class="ancho">
  <div><p><b>{{autor}}</b></p><p>Hecho con Python, pandas y scikit-learn. Gráficos en SVG propios, sin librerías.</p>{{enlaces}}</div>
  <div><p>Datos hasta el {{hasta}}. Página generada el {{generado}}.</p><p>Proyecto de análisis. No es una recomendación para apostar. Los escudos pertenecen a sus clubes; Bet365 es una marca de su propietario y no tiene relación con este trabajo.</p></div>
</div></footer>
'''

## 3.3 Gráficos y controles

In [8]:
# =============================================================================
#  CELDA · GRÁFICOS Y CONTROLES
# =============================================================================
CODIGO = r'''const D = JSON.parse(document.getElementById('datos').textContent);
const $ = s => document.querySelector(s), NS = 'http://www.w3.org/2000/svg', SIG = ['1', 'X', '2'];
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();
const C = { m: css('--modelo'), b: css('--casa'), tiza: css('--tiza'), t2: css('--tiza-2'), c: css('--cesped'), c2: css('--cesped-2'), c3: css('--cesped-3') };
const nf = (v, d = 0) => Math.abs(v).toLocaleString('es-ES', { minimumFractionDigits: d, maximumFractionDigits: d, useGrouping: 'always' });
const num = (v, d = 0) => (v < 0 && Number(Math.abs(v).toFixed(d)) !== 0 ? '−' : '') + nf(v, d);
const sg = (v, d = 2) => { const c = Number(Math.abs(v).toFixed(d)) === 0; return (c ? '' : v < 0 ? '−' : '+') + nf(v, d); };
const temp = s => s + '/' + String((s + 1) % 100).padStart(2, '0');
const cortaT = s => String(s % 100).padStart(2, '0') + '/' + String((s + 1) % 100).padStart(2, '0');
const MES = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre'];
const DIA = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado'];
const fecha = s => { const t = s.replace(/-/g, ''); return new Date(+t.slice(0, 4), +t.slice(4, 6) - 1, +t.slice(6, 8)); };
const diaMes = f => f.getDate() + ' de ' + MES[f.getMonth()];
const EQ = Object.fromEntries(D.equipos.map((e, i) => [e.k, { ...e, i }]));
const ini = n => n.split(' ').map(x => x[0]).join('').slice(0, 2).toUpperCase() + (n.includes(' ') ? '' : n.slice(1, 3).toUpperCase());
const escudo = e => e.e ? `<img class="escudo" src="${e.e}" alt="">` : `<span class="escudo sin" aria-hidden="true">${ini(e.n).slice(0, 3)}</span>`;
const svg = (w, h) => { const s = document.createElementNS(NS, 'svg'); s.setAttribute('viewBox', `0 0 ${w} ${h}`); s.setAttribute('width', w); s.setAttribute('height', h); s.setAttribute('aria-hidden', 'true'); return s; };
const E = (p, t, a, txt) => { const e = document.createElementNS(NS, t); for (const k in a) e.setAttribute(k, a[k]); if (txt !== undefined) e.textContent = txt; p.appendChild(e); return e; };
const T = (p, x, y, txt, a = {}) => E(p, 'text', { x, y, fill: C.t2, 'font-size': 13, ...a }, txt);

function globo(cont) {
  let g = cont.querySelector('.globo');
  if (!g) { g = document.createElement('div'); g.className = 'globo'; cont.appendChild(g); }
  return { pon(html, x, y) { g.innerHTML = html; g.classList.add('ver'); const w = g.offsetWidth, W = cont.clientWidth; let l = x + 16; if (l + w > W) l = x - w - 16; g.style.transform = `translate(${Math.max(0, Math.min(l, W - w))}px, ${Math.max(0, y)}px)`; }, quita() { g.classList.remove('ver'); } };
}
const linea = (n, v) => `<div><span>${n}</span><b style="margin:0">${v}</b></div>`;
function pasos(lo, hi, n) { const r = hi - lo, p = Math.pow(10, Math.floor(Math.log10(r / n))), k = [1, 2, 2.5, 5, 10].find(k => r / (k * p) <= n) * p; const o = []; for (let v = Math.ceil(lo / k) * k; v <= hi + 1e-9; v += k) o.push(Math.abs(v) < 1e-9 ? 0 : v); return o; }

/* dos curvas acumuladas sobre el césped; `franjas` = [inicio, fin, rótulo] en unidades de x */
function curvas(cont, { x, m, b, franjas, alto, rot, globoTxt, franjaCada = 1 }) {
  const W = Math.max(320, cont.clientWidth), est = W < 620, H = est ? Math.min(alto, 300) : alto, dcha = est ? 62 : 98, izq = est ? 40 : 54, arr = 14, aba = 34;
  const X0 = x[0], X1 = x[x.length - 1], lo0 = Math.min(0, ...m, ...b), hi0 = Math.max(0, ...m, ...b), am = (hi0 - lo0) || 1, lo = lo0 - am * .05, hi = hi0 + am * .07;
  const px = v => izq + (v - X0) / (X1 - X0 || 1) * (W - izq - dcha), py = v => arr + (hi - v) / (hi - lo) * (H - arr - aba);
  cont.querySelector('svg')?.remove();
  const s = svg(W, H); cont.prepend(s);
  franjas.forEach(([a, z, r], i) => {
    E(s, 'rect', { x: px(a), y: arr, width: Math.max(0, px(z) - px(a)), height: H - arr - aba, fill: i % 2 ? C.c2 : C.c3, opacity: i % 2 ? 1 : .55 });
    const fin = i === franjas.length - 1 && px(z) - px(a) < 40 && franjas.length > 1; if (i % (est ? Math.max(franjaCada, 4) : franjaCada) === 0) T(s, fin ? px(a) + 2 : (px(a) + px(z)) / 2, H - 11, r, { 'text-anchor': fin ? 'start' : 'middle', 'font-size': 12 });
  });
  pasos(lo, hi, est ? 4 : 6).forEach(v => { if (v) E(s, 'line', { x1: izq, x2: W - dcha, y1: py(v), y2: py(v), stroke: C.tiza, 'stroke-opacity': .1 }); T(s, izq - 10, py(v) + 4, (v > 0 ? '+' : '') + num(v), { 'text-anchor': 'end', 'font-size': 12.5 }); });
  E(s, 'line', { x1: izq, x2: W - dcha, y1: py(0), y2: py(0), stroke: C.tiza, 'stroke-width': 2, 'stroke-opacity': .85 });
  const ruta = v => v.map((q, i) => (i ? 'L' : 'M') + px(x[i]).toFixed(1) + ' ' + py(q).toFixed(1)).join('');
  E(s, 'path', { d: ruta(b), fill: 'none', stroke: C.b, 'stroke-width': 2, 'stroke-dasharray': '6 4', 'stroke-linejoin': 'round' });
  E(s, 'path', { d: ruta(m), fill: 'none', stroke: C.m, 'stroke-width': 3, 'stroke-linejoin': 'round', 'stroke-linecap': 'round' });
  const n = x.length - 1; let ym = py(m[n]), yb = py(b[n]); const dm = Math.abs(ym - yb) < 40 ? (40 - Math.abs(ym - yb)) / 2 : 0, sube = ym <= yb ? -1 : 1;
  [[b, C.b, 'Bet365', yb - sube * dm], [m, C.m, 'Modelo', ym + sube * dm]].forEach(([v, col, r, y]) => {
    E(s, 'circle', { cx: px(x[n]), cy: py(v[n]), r: 5.5, fill: col, stroke: C.c, 'stroke-width': 2 });
    y = Math.max(arr + 10, Math.min(y, H - aba - 12));
    T(s, px(x[n]) + 11, y + 1, sg(v[n], est ? 0 : 1), { fill: C.tiza, 'font-size': est ? 14 : 17, 'font-weight': 700 }); if (!est) T(s, px(x[n]) + 11, y + 17, r, { 'font-size': 12.5 });
  });
  const guia = E(s, 'line', { y1: arr, y2: H - aba, stroke: C.tiza, 'stroke-opacity': 0 }), pm = E(s, 'circle', { r: 5, fill: C.m, stroke: C.c, 'stroke-width': 2, opacity: 0 }), pb = E(s, 'circle', { r: 5, fill: C.b, stroke: C.c, 'stroke-width': 2, opacity: 0 });
  const zona = E(s, 'rect', { x: izq, y: arr, width: W - izq - dcha, height: H - arr - aba, fill: 'transparent' }), g = globo(cont);
  const mueve = e => { const r = s.getBoundingClientRect(), v = X0 + ((e.clientX - r.left) * W / r.width - izq) / (W - izq - dcha) * (X1 - X0); let i = 0, j = x.length - 1; while (j - i > 1) { const k = (i + j) >> 1; x[k] < v ? i = k : j = k; } i = Math.abs(x[i] - v) < Math.abs(x[j] - v) ? i : j;
    guia.setAttribute('x1', px(x[i])); guia.setAttribute('x2', px(x[i])); guia.setAttribute('stroke-opacity', .5); [[pm, m], [pb, b]].forEach(([o, q]) => { o.setAttribute('cx', px(x[i])); o.setAttribute('cy', py(q[i])); o.setAttribute('opacity', 1); });
    g.pon(`<b>${rot(i)}</b>` + linea('Modelo', sg(m[i], 1) + ' u') + linea('Bet365', sg(b[i], 1) + ' u') + (globoTxt ? globoTxt(i) : ''), px(x[i]) * r.width / W, arr + 6); };
  const sale = () => { g.quita(); guia.setAttribute('stroke-opacity', 0); pm.setAttribute('opacity', 0); pb.setAttribute('opacity', 0); };
  zona.addEventListener('pointermove', mueve); zona.addEventListener('pointerdown', mueve); zona.addEventListener('pointerleave', sale);
}

function portada() {
  let n = 0; const x = [0], m = [0], b = [0], r = [null], fr = [];
  D.curva.forEach(([s, j, k, a, c]) => { if (!fr.length || fr[fr.length - 1][3] !== s) fr.push([n, n, cortaT(s), s]); n += k; fr[fr.length - 1][1] = n; x.push(n); m.push(a); b.push(c); r.push([s, j]); });
  curvas($('#g-curva'), { x, m, b, franjas: fr, alto: 430, rot: i => i ? `${temp(r[i][0])}, jornada ${r[i][1]}` : 'Inicio' });
}

function ventaja() {
  const cont = $('#g-ventaja'), W = Math.max(320, cont.clientWidth), est = W < 620, H = est ? 280 : 320, izq = est ? 34 : 46, dcha = 8, arr = 26, aba = 30, t = D.temporadas;
  const lo = Math.min(0, ...t.map(q => q.ventaja)) - 3, hi = Math.max(...t.map(q => q.ventaja)) + 2, py = v => arr + (hi - v) / (hi - lo) * (H - arr - aba), paso = (W - izq - dcha) / t.length, an = Math.min(34, paso - (est ? 3 : 8));
  cont.querySelector('svg')?.remove(); const s = svg(W, H), g = globo(cont); cont.prepend(s);
  pasos(lo, hi, 6).forEach(v => { if (v) E(s, 'line', { x1: izq, x2: W - dcha, y1: py(v), y2: py(v), stroke: C.tiza, 'stroke-opacity': .1 }); T(s, izq - 8, py(v) + 4, (v > 0 ? '+' : '') + num(v), { 'text-anchor': 'end', 'font-size': 12.5 }); });
  t.forEach((q, i) => {
    const cx = izq + paso * (i + .5), y0 = py(0), y1 = py(q.ventaja), sube = q.ventaja >= 0, r = Math.min(4, Math.abs(y1 - y0)), a = cx - an / 2, z = cx + an / 2, k = sube ? r : -r;
    const p = E(s, 'path', { d: `M${a} ${y0}L${a} ${y1 + k}Q${a} ${y1} ${a + r} ${y1}L${z - r} ${y1}Q${z} ${y1} ${z} ${y1 + k}L${z} ${y0}Z`, fill: sube ? C.m : C.b });
    if (!est || Math.abs(q.ventaja) > 5 || i === t.length - 1) T(s, cx, sube ? y1 - 8 : y1 + 17, sg(q.ventaja, 1), { 'text-anchor': 'middle', fill: C.tiza, 'font-size': est ? 11 : 13, 'font-weight': 600 });
    if (!est || i % 4 === 0) T(s, cx, H - 8, cortaT(q.s), { 'text-anchor': 'middle', 'font-size': 12 });
    const z2 = E(s, 'rect', { x: izq + paso * i, y: 0, width: paso, height: H, fill: 'transparent' });
    const ver = () => { p.setAttribute('opacity', .8); const k2 = s.getBoundingClientRect().width / W; g.pon(`<b>${q.t}</b>` + linea('Modelo', sg(q.roi, 1) + ' %') + linea('Bet365', sg(q.roi_mk, 1) + ' %') + linea('Ventaja', sg(q.ventaja, 1)) + linea('Se aparta en', q.dis + ' de ' + q.n), cx * k2, 10); };
    z2.addEventListener('pointerenter', ver); z2.addEventListener('pointerdown', ver); z2.addEventListener('pointerleave', () => { p.setAttribute('opacity', 1); g.quita(); });
  });
  E(s, 'line', { x1: izq, x2: W - dcha, y1: py(0), y2: py(0), stroke: C.tiza, 'stroke-width': 2, 'stroke-opacity': .85 });
}

function fiabilidad() {
  const cont = $('#g-fiab'), W = Math.max(300, cont.clientWidth), H = Math.min(360, W * .62), izq = 48, dcha = 16, arr = 10, aba = 44, px = v => izq + v * (W - izq - dcha), py = v => arr + (1 - v) * (H - arr - aba);
  cont.querySelector('svg')?.remove(); const s = svg(W, H), g = globo(cont); cont.prepend(s);
  [0, .25, .5, .75, 1].forEach(v => { E(s, 'line', { x1: px(0), x2: px(1), y1: py(v), y2: py(v), stroke: C.tiza, 'stroke-opacity': .1 }); T(s, izq - 8, py(v) + 4, num(100 * v) + ' %', { 'text-anchor': 'end', 'font-size': 12 }); T(s, px(v), H - 24, num(100 * v) + ' %', { 'text-anchor': v === 1 ? 'end' : 'middle', 'font-size': 12 }); });
  T(s, px(.5), H - 5, 'Probabilidad pronosticada', { 'text-anchor': 'middle', 'font-size': 12.5 }); T(s, px(0) + 8, arr + 12, 'Frecuencia real', { 'font-size': 12.5 });
  E(s, 'line', { x1: px(0), y1: py(0), x2: px(1), y2: py(1), stroke: C.t2, 'stroke-width': 1 });
  [['b', C.b, 'Bet365', '6 4', 2], ['m', C.m, 'Modelo', '', 3]].forEach(([k, col, r, da, an]) => {
    const q = D.fiab[k]; E(s, 'path', { d: q.map((z, i) => (i ? 'L' : 'M') + px(z[0]) + ' ' + py(z[1])).join(''), fill: 'none', stroke: col, 'stroke-width': an, 'stroke-dasharray': da });
    q.forEach(z => { E(s, 'circle', { cx: px(z[0]), cy: py(z[1]), r: 4.5, fill: col, stroke: C.c, 'stroke-width': 2 }); const o = E(s, 'circle', { cx: px(z[0]), cy: py(z[1]), r: 12, fill: 'transparent' }); const ver = () => g.pon(`<b>${r}</b>` + linea('Pronosticaba', num(100 * z[0], 1) + ' %') + linea('Ocurrió', num(100 * z[1], 1) + ' %') + linea('Casos', num(z[2])), px(z[0]) * s.getBoundingClientRect().width / W, py(z[1]) - 40); o.addEventListener('pointerenter', ver); o.addEventListener('pointerdown', ver); o.addEventListener('pointerleave', () => g.quita()); });
  });
}

function tablaTemporadas() {
  const c = (v, o, d, s, f = num) => `<td${(s === 'alto' ? v > o : v < o) ? ' class="mejor"' : ''}>${f(v, d)}</td>`;
  $('#t-temporadas').innerHTML = `<table class="t"><thead><tr><th>Temporada</th><th>Partidos</th><th>Acierto<br>modelo</th><th>Acierto<br>Bet365</th><th>Rentab.<br>modelo</th><th>Rentab.<br>Bet365</th><th>Unidades<br>modelo</th><th>Unidades<br>Bet365</th><th>Pérdida log.<br>modelo</th><th>Pérdida log.<br>Bet365</th></tr></thead><tbody>` +
    [...D.temporadas].reverse().map(q => `<tr><td>${q.t}</td><td>${q.n}</td>${c(q.acierto, q.acierto_mk, 1, 'alto')}${c(q.acierto_mk, q.acierto, 1, 'alto')}${c(q.roi, q.roi_mk, 1, 'alto', sg)}${c(q.roi_mk, q.roi, 1, 'alto', sg)}${c(q.unidades, q.unidades_mk, 1, 'alto', sg)}${c(q.unidades_mk, q.unidades, 1, 'alto', sg)}${c(q.ll, q.ll_mk, 3, 'bajo')}${c(q.ll_mk, q.ll, 3, 'bajo')}</tr>`).join('') + '</tbody></table>';
}

function confusion() {
  const n = ['Local', 'Empate', 'Visitante'], M = D.confusion, mx = Math.max(...M.flat());
  let h = '<span></span>' + n.map(q => `<span class="co">Pronostica<br>${q.toLowerCase()}</span>`).join('');
  M.forEach((f, i) => { const t = f.reduce((a, b) => a + b, 0); h += `<span class="fi">Sale ${n[i].toLowerCase()}</span>` + f.map((v, j) => { const k = v / mx; return `<span class="cel" style="background:color-mix(in srgb, var(--cesped) ${Math.round(6 + 94 * Math.pow(k, .75))}%, var(--blanco));color:${k > .6 ? 'var(--tiza)' : 'var(--tinta)'}${i === j ? ';box-shadow:inset 0 0 0 2px var(--modelo)' : ''}">${num(v)}<small>${num(100 * v / t)} %</small></span>`; }).join(''); });
  $('#confusion').innerHTML = h;
}

function discrepancias() {
  const n = { H: 'Local (1)', D: 'Empate (X)', A: 'Visitante (2)' };
  $('#t-dis').innerHTML = '<table class="t"><thead><tr><th>Cuando pronostica</th><th>Partidos</th><th>Acierta<br>modelo</th><th>Acierta<br>favorito</th></tr></thead><tbody>' + D.dis_signo.map(([s, k, a, b]) => `<tr><td>${n[s]}</td><td>${num(k)}</td><td${a > b ? ' class="mejor"' : ''}>${num(100 * a / (k || 1), 1)} %</td><td${b > a ? ' class="mejor"' : ''}>${num(100 * b / (k || 1), 1)} %</td></tr>`).join('') + '</tbody></table>';
}

function boleto() {
  const c = $('#boleto'), P = D.proxima;
  if (!P.length) { c.innerHTML = '<div class="boleto" style="grid-template-columns:1fr"><p class="vacio">Ahora mismo no hay partidos por jugar en los datos. El pronóstico aparece aquí cuando se prepara la siguiente jornada.</p></div>'; return; }
  const dif = P.filter(q => q.y !== q.fav).length, f0 = fecha(P[0].f), f1 = fecha(P[P.length - 1].f), js = [...new Set(P.map(q => q.j))];
  const filas = P.map(q => {
    const l = EQ[q.l], v = EQ[q.v], o = [...q.p].sort((a, b) => b - a), igual = o[0] - o[1] < .02, et = [q.y !== q.fav ? `<span class="etiqueta">Se aparta del favorito</span>` : '', igual ? '<span class="etiqueta ojo">Muy igualado</span>' : ''].filter(Boolean).join(' '), f = fecha(q.f);
    const cas = q.p.map((p, i) => `<span class="casilla${i === q.y ? ' elegida' : ''}" style="--w:${(100 * p).toFixed(1)}%"><span class="s">${SIG[i]}</span><span class="p">${num(100 * p)} %</span><span class="k">Bet365 ${num(100 * q.m[i])}</span></span>`).join('');
    const ficha = q.ficha.map(([r, d, h, a, nh, na]) => `<div class="met"><span class="rot">${r}</span><span class="v">${num(h, d)}</span><span class="b i${h > a ? ' gana' : ''}"><span style="width:${(100 * nh).toFixed(1)}%"></span></span><span class="b${a > h ? ' gana' : ''}"><span style="width:${(100 * na).toFixed(1)}%"></span></span><span class="v">${num(a, d)}</span></div>`).join('');
    return `<details class="fila"><summary aria-label="${l.n} contra ${v.n}. Pronóstico ${SIG[q.y]} con ${num(100 * q.p[q.y])} por ciento. Ver comparación de los equipos"><span class="duelo"><span class="eq">${escudo(l)}<span>${l.n}</span></span><span class="eq">${escudo(v)}<span>${v.n}</span></span><span class="et2" style="display:none"><span class="chico">${DIA[f.getDay()]} ${f.getDate()}</span> ${et}</span></span><span class="tres">${cas}</span><span class="et"><span class="chico" style="display:block;margin-bottom:4px">${DIA[f.getDay()]} ${f.getDate()}</span>${et}</span><span class="flecha" aria-hidden="true"></span></summary>` +
      `<div class="ficha"><div class="quien"><span>${l.n}</span><span>${v.n}</span></div>${ficha}<p class="chico num">Cuotas de Bet365: ${q.o.map((z, i) => SIG[i] + ' a ' + num(z, 2)).join(', ')}. Los promedios son de los 7 últimos partidos de cada equipo.</p></div></details>`;
  }).join('');
  c.innerHTML = `<div class="boleto"><div class="matriz"><div>Jornada<div class="grande num">${js.join('·')}</div></div><div>${temp(P[0].s)}</div><div class="lineas" aria-hidden="true"><svg viewBox="0 0 100 154" stroke-width="1"><rect x="1" y="1" width="98" height="152"/><path d="M1 77H99M22 1V25H78V1M37 1V10H63V1M22 153V129H78V153M37 153V144H63V153M40 25A13 13 0 0 0 60 25M40 129A13 13 0 0 1 60 129"/><circle cx="50" cy="77" r="14"/></svg></div><dl><div><dt>Fechas</dt><dd>${f0.getMonth() === f1.getMonth() ? f0.getDate() : diaMes(f0)} a ${diaMes(f1)}</dd></div><div><dt>Partidos</dt><dd>${P.length}</dd></div><div><dt>Se aparta de Bet365</dt><dd>${dif === 0 ? 'En ninguno' : 'En ' + dif}</dd></div></dl></div><div class="filas"><div class="fila-cab"><span>Partido</span><span class="tres"><span>Local</span><span>Empate</span><span>Visitante</span></span><span></span><span></span></div>${filas}</div></div>`;
}

const XT = {}; D.partidos.forEach(q => { ((XT[q[0]] ??= {})[q[1]] ??= []).push(q); });
const ben = (q, k) => k === q[7] ? q[13 + k] - 1 : -1;
let S, J;
function dibujaTemp() {
  const js = Object.keys(XT[S]).map(Number).sort((a, b) => a - b), x = [0], m = [0], b = [0]; let a = 0, c = 0;
  js.forEach(j => { XT[S][j].forEach(q => { a += ben(q, q[8]); c += ben(q, q[9]); }); x.push(j); m.push(a); b.push(c); });
  const ult = js[js.length - 1], fr = []; for (let j = 0; j < ult; j++) fr.push([j, j + 1, String(j + 1)]);
  curvas($('#g-temp'), { x, m, b, franjas: fr, alto: 320, franjaCada: ult > 20 ? 2 : 1, rot: i => i ? 'Jornada ' + x[i] : 'Inicio' });
}
function explora(reJ) {
  const t = D.temporadas.find(q => q.s === S), js = Object.keys(XT[S]).map(Number).sort((a, b) => a - b), sj = $('#sel-j');
  if (reJ) { sj.innerHTML = js.map(j => `<option value="${j}">Jornada ${j}</option>`).join(''); if (!js.includes(J)) J = js[js.length - 1]; $('#exp-tit').textContent = 'Temporada ' + t.t; $('#exp-sub').textContent = `Beneficio acumulado, 1 unidad por partido. Modelo: ${t.ok} aciertos de ${t.n} (${num(t.acierto, 1)} %) y ${sg(t.unidades, 1)} unidades. Bet365: ${t.ok_mk} aciertos (${num(t.acierto_mk, 1)} %) y ${sg(t.unidades_mk, 1)}.`; dibujaTemp(); }
  sj.value = J; $('#sel-t').value = S; $('#ant').disabled = J === js[0]; $('#sig').disabled = J === js[js.length - 1];
  const L = XT[S][J], s = k => L.reduce((a, q) => a + ben(q, q[k]), 0), ok = k => L.filter(q => q[k] === q[7]).length;
  $('#jor-tit').textContent = `Jornada ${J} de ${t.t}`;
  $('#jor-res').innerHTML = `<span>Modelo <b>${ok(8)} de ${L.length}</b>, <b>${sg(s(8))}</b> unidades</span><span>Bet365 <b>${ok(9)} de ${L.length}</b>, <b>${sg(s(9))}</b> unidades</span>`;
  $('#jor-lista').innerHTML = '<div class="par cab"><span style="text-align:right">Local</span><span style="text-align:center">Resultado</span><span>Visitante</span><span style="text-align:center">Modelo</span><span style="text-align:center">Favorito</span><span style="text-align:right">Unidades</span></div>' +
    L.map(q => { const l = D.equipos[q[3]], v = D.equipos[q[4]], k = q[8], si = k === q[7], b = ben(q, k); return `<div class="par"><span class="eq l"><span>${l.n}</span>${escudo(l)}</span><span class="res">${q[5] < 0 ? SIG[q[7]] : q[5] + '–' + q[6]}</span><span class="eq">${escudo(v)}<span>${v.n}</span></span><span class="cel m"><span class="pico${si ? ' si' : ''}">${si ? '✓' : '✗'} ${SIG[k]}<span class="x">${num(q[10 + k] / 10)} %</span></span></span><span class="cel c">${SIG[q[9]]}<span class="chico"> a ${num(q[13 + q[9]], 2)}</span></span><span class="ben">${sg(b)}</span></div>`; }).join('');
}
function explorador() {
  const st = $('#sel-t'); st.innerHTML = [...D.temporadas].reverse().map(q => `<option value="${q.s}">${q.t}</option>`).join('');
  S = D.temporadas[D.temporadas.length - 1].s; J = Math.max(...Object.keys(XT[S]).map(Number)); explora(true);
  st.onchange = () => { S = +st.value; J = -1; explora(true); }; $('#sel-j').onchange = e => { J = +e.target.value; explora(false); };
  const mueve = d => { const js = Object.keys(XT[S]).map(Number).sort((a, b) => a - b), i = js.indexOf(J) + d; if (js[i] !== undefined) { J = js[i]; explora(false); } };
  $('#ant').onclick = () => mueve(-1); $('#sig').onclick = () => mueve(1);
}

function walkforward() {
  const cont = $('#g-wf'), M = D.modelo; if (!M) { cont.closest('.panel').hidden = true; return; }
  const W = Math.max(320, cont.clientWidth), est = W < 620, fil = 4, al = est ? 26 : 30, sep = est ? 12 : 14, arr = est ? 50 : 40, H = arr + fil * (al + sep) + 34, V = M.n_seasons_window * M.season_size, R = M.season_size;
  cont.querySelector('svg')?.remove(); const s = svg(W, H); cont.prepend(s);
  const util = W - (est ? 4 : 10), av = util * .035, pz = est ? 24 : 34, tot = util - av * (fil - 1) - pz - 4, viejo = tot * (V - R) / V, nuevo = tot * R / V;
  for (let i = 0; i < fil; i++) {
    const x = i * av, y = arr + i * (al + sep), op = 1 - i * .0;
    E(s, 'rect', { x, y, width: viejo - 2, height: al, rx: 4, fill: C.c2, stroke: C.t2, 'stroke-opacity': .45, opacity: op });
    E(s, 'rect', { x: x + viejo, y, width: nuevo - 2, height: al, rx: 4, fill: C.b, opacity: op });
    E(s, 'rect', { x: x + viejo + nuevo + 2, y, width: pz, height: al, rx: 4, fill: C.m, opacity: op });
    if (!i) { T(s, x + 12, y + al / 2 + 5, num(V - R) + (est ? ' partidos' : ' partidos, peso ' + num(M.older_weight)), { fill: C.tiza, 'font-size': est ? 12 : 14 }); T(s, x + viejo + (nuevo - 2) / 2, y + al / 2 + 5, est ? num(R) : num(R) + ', peso ' + num(M.recent_weight), { fill: C.c3, 'font-size': est ? 12 : 14, 'font-weight': 700, 'text-anchor': 'middle' }); }
  }
  T(s, 0, arr - 14, 'Entrenamiento', { 'font-size': 13 }); T(s, viejo + nuevo + 2 + pz, arr - 14, 'Fecha pronosticada', { 'font-size': 13, 'text-anchor': 'end', fill: C.m });
  T(s, 0, H - 8, 'Fechas anteriores', { 'font-size': 12.5 }); T(s, W - 4, H - 8, 'Fechas posteriores →', { 'font-size': 12.5, 'text-anchor': 'end' });
}

let VS = 0;
function pintaSenal() {
  const cont = $('#g-senal'), v = D.senales[VS]; if (!v) return;
  const W = Math.max(280, cont.clientWidth), est = W < 480, al = 25, n = v.x.length, izq = est ? 112 : 136, dcha = 52, arr = 24, H = arr + n * al + 6;
  const vs = v.x.map(q => q[1]), lo = Math.min(0, ...vs), hi = Math.max(0, ...vs), hueco = lo < 0 ? 46 : 0, px = q => izq + hueco + (q - lo) / (hi - lo || 1) * (W - izq - dcha - hueco);
  cont.querySelector('svg')?.remove(); const s = svg(W, H); cont.prepend(s);
  T(s, W, 12, v.u, { 'text-anchor': 'end', 'font-size': 12.5 });
  E(s, 'line', { x1: px(0), x2: px(0), y1: arr - 4, y2: H - 4, stroke: C.tiza, 'stroke-opacity': .45 });
  v.x.forEach(([k, q], i) => {
    const y = arr + i * al, e = EQ[k] || { n: k }, a = px(0), w = Math.max(1.5, Math.abs(px(q) - a)), pos = q >= 0;
    if (e.e) E(s, 'image', { href: e.e, x: 0, y: y + 2, width: 19, height: 19 });
    T(s, 27, y + 16, e.n, { fill: C.tiza, 'font-size': est ? 12.5 : 14, 'font-weight': i === 0 || i === n - 1 ? 700 : 400 });
    E(s, 'rect', { x: pos ? a : a - w, y: y + 5, width: w, height: 13, rx: 3, fill: pos ? C.m : C.b });
    T(s, pos ? a + w + 7 : a - w - 7, y + 16, sg(q, v.dec), { fill: C.tiza, 'font-size': 12, 'text-anchor': pos ? 'start' : 'end' });
  });
  const nom = k => (EQ[k] || { n: k }).n;
  $('#senal-txt').innerHTML = `<h4>${v.n}</h4><p>${v.d}</p><p class="formula">${v.f}</p><p class="hoy">${v.hoy.replace('{a}', nom(v.x[0][0])).replace('{z}', nom(v.x[n - 1][0]))}</p>`;
}
function senales() {
  const c = $('#senal-tabs'); if (!D.senales.length) { c.closest('.panel').hidden = true; return; }
  c.innerHTML = D.senales.map((v, i) => `<button type="button" role="tab" aria-selected="${!i}" aria-controls="senal-txt" tabindex="${i ? -1 : 0}">${v.q}</button>`).join('');
  const bs = [...c.children], elige = i => { VS = i; bs.forEach((b, j) => { b.setAttribute('aria-selected', i === j); b.tabIndex = i === j ? 0 : -1; }); pintaSenal(); };
  bs.forEach((b, i) => { b.onclick = () => elige(i); b.onkeydown = e => { const d = { ArrowDown: 1, ArrowRight: 1, ArrowUp: -1, ArrowLeft: -1 }[e.key]; if (d) { e.preventDefault(); const j = (i + d + bs.length) % bs.length; elige(j); bs[j].focus(); } }; });
}
function embudo() {
  const n = D.cadena.variables, k = D.n_vars, si = new Set(); if (!k) { $('#cuadros').closest('.panel').hidden = true; return; }
  for (let i = 0; i < k; i++) si.add(Math.floor((i + .5) * n / k) + [-9, 6, -4, 9, 3, -7, 8, -2][i % 8]);
  $('#cuadros').innerHTML = Array.from({ length: n }, (_, i) => `<i${si.has(i) ? ' class="si"' : ''}></i>`).join('');
}

function dibuja() { portada(); ventaja(); fiabilidad(); dibujaTemp(); walkforward(); pintaSenal(); }
boleto(); discrepancias(); tablaTemporadas(); confusion(); senales(); embudo(); explorador(); dibuja();
let an = innerWidth, tmr; addEventListener('resize', () => { if (innerWidth === an) return; an = innerWidth; clearTimeout(tmr); tmr = setTimeout(dibuja, 120); });
document.fonts?.ready.then(dibuja);
'''

# **4. Página**

In [9]:
# =============================================================================
#  CELDA · MONTAJE, CONTROL Y ESCRITURA
# =============================================================================
MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
LETRAS = {10: "Diez", 11: "Once", 12: "Doce", 13: "Trece", 14: "Catorce", 15: "Quince", 16: "Dieciséis", 17: "Diecisiete", 18: "Dieciocho", 19: "Diecinueve", 20: "Veinte"}


def es(v, dec=0, signo=False):
    c = round(abs(v), dec) == 0
    t = f"{abs(v):,.{dec}f}".replace(",", "·").replace(".", ",").replace("·", ".")
    return ("" if c else "−" if v < 0 else "+" if signo else "") + t


def dia(s):
    f = pd.Timestamp(s)
    return f"{f.day} de {MESES[f.month - 1]} de {f.year}"


T, X, K = DATOS["total"], DATOS["discrepa"], DATOS["cadena"]
ult = temps[-1]
jor_act = int(ap.loc[ap["Season"] == ult["s"], "Matchday"].max())
limpio = ap[(ap["Season"] > LIMPIO_DESDE[0]) | ((ap["Season"] == LIMPIO_DESDE[0]) & (ap["Matchday"] >= LIMPIO_DESDE[1]))]
LM = bloque(limpio) if len(limpio) else None
W_ = MODELO["n_seasons_window"] * MODELO["season_size"] if MODELO else 0
if proxima:
    a, b = pd.Timestamp(proxima[0]["f"]), pd.Timestamp(proxima[-1]["f"])
    rango = (f"Del {a.day} al {b.day} de {MESES[b.month - 1]}" if a.month == b.month else f"Del {a.day} de {MESES[a.month - 1]} al {b.day} de {MESES[b.month - 1]}") if a != b else f"El {a.day} de {MESES[a.month - 1]}"
    js = sorted({x["j"] for x in proxima})
    tit_j = "El pronóstico de la jornada " + " y ".join(map(str, js))
    txt_j = rango + ". Cada casilla lleva la probabilidad que el modelo da a ese signo y, debajo, la de Bet365. La verde es el pronóstico. Pulsa un partido para comparar los dos equipos."
else:
    tit_j, txt_j = "Próxima jornada", "El pronóstico se publica antes de cada jornada."
enl = "".join(f'<a href="{u}" target="_blank" rel="noopener">{n}</a>' for n, u in ENLACES.items() if u)


def mini_curva(a, b, js, an=380, al=150, m=(10, 50, 20, 6)):
    # unidades acumuladas en la temporada: modelo (amarillo) y favorito de Bet365 (discontinua)
    lo, hi = min(0, *a, *b), max(0, *a, *b)
    if hi - lo < 1:
        hi = lo + 1
    sx = lambda i: m[3] + (an - m[1] - m[3]) * i / max(1, len(a) - 1)
    sy = lambda v: m[0] + (al - m[0] - m[2]) * (hi - v) / (hi - lo)
    pts = lambda s: " ".join(f"{sx(i):.1f},{sy(v):.1f}" for i, v in enumerate(s))
    ya, yb = sy(a[-1]) + 4, sy(b[-1]) + 4
    if abs(ya - yb) < 13:                       # que las dos etiquetas finales no se pisen
        ya, yb = (ya - 7, yb + 7) if a[-1] >= b[-1] else (ya + 7, yb - 7)
    return (f'<div class="grafico mini"><svg viewBox="0 0 {an} {al}" role="img" aria-label="Unidades acumuladas en la temporada: '
            f'modelo {es(a[-1], 1, True)}, favorito de Bet365 {es(b[-1], 1, True)}.">'
            f'<line x1="{m[3]}" x2="{an - m[1]}" y1="{sy(0):.1f}" y2="{sy(0):.1f}" class="cero"/>'
            f'<polyline points="{pts(b)}" class="l-casa"/><polyline points="{pts(a)}" class="l-modelo"/>'
            f'<text x="{an - m[1] + 6}" y="{ya:.1f}" class="v-modelo">{es(a[-1], 1, True)}</text>'
            f'<text x="{an - m[1] + 6}" y="{yb:.1f}" class="v-casa">{es(b[-1], 1, True)}</text>'
            f'<text x="{m[3]}" y="{al - 4}" class="eje">Jornada {js[0]}</text>'
            f'<text x="{an - m[1]}" y="{al - 4}" text-anchor="end" class="eje">Jornada {js[-1]}</text></svg></div>')


seccion_chat = nav_chat = ""
if CHAT:
    _act = ap[ap["Season"] == ult["s"]]
    _b = bloque(_act)
    _pj = porjor[porjor["Season"] == ult["s"]].reset_index(drop=True)
    _curva = mini_curva(_pj["m"].cumsum().tolist(), _pj["b"].cumsum().tolist(), _pj["Matchday"].astype(int).tolist())
    nav_chat = '<a href="#chat">Pregunta a los datos</a>'
    seccion_chat = f'''<section class="campo producto" id="chat"><div class="ancho">
  <div class="producto-rej">
    <div class="producto-txt">
      <p class="ante">Nuevo · un agente de IA con toda la base</p>
      <h2>Pregunta a los datos</h2>
      <p class="entrada">Un chat con un agente que tiene acceso a todos los datos del proyecto: {es(K["partidos"])} partidos desde {K["desde"]}, con resultados, cuotas de veinte casas, xG, Elo, valor de plantilla, la estadística completa de SofaScore y los pronósticos del modelo. Le preguntas como a una persona y responde consultando los datos.</p>
      <ul class="ventajas">
        <li><b>Cifras de la base, no de memoria.</b> Cada respuesta enseña las consultas que ha hecho para que se puedan comprobar.</li>
        <li><b>Tablas y gráficos.</b> Agrega, compara y dibuja lo que le pidas; las tablas se descargan en CSV.</li>
        <li><b>Funciona dentro de Claude.</b> Hace falta una cuenta de Claude, y las preguntas cuentan para el plan de quien pregunta.</li>
      </ul>
      <p><a class="cta" href="{CHAT}" target="_blank" rel="noopener">Abrir el chat <span aria-hidden="true">↗</span></a></p>
      <p class="cta-nota">Se abre en claude.ai, donde hay que iniciar sesión.</p>
    </div>
    <figure class="ventana">
      <figcaption>Vista previa · una respuesta de ejemplo, con los datos de hoy</figcaption>
      <div class="burbuja yo">¿Cómo le va al modelo esta temporada frente al favorito de Bet365?</div>
      <div class="burbuja agente">
        <span class="paso-chat">Consulta · predicciones · {_b["n"]} partidos de {ult["t"]}</span>
        <p>En {ult["t"]} el modelo lleva <b>{_b["ok"]} aciertos de {_b["n"]}</b> y <b>{es(_b["unidades"], 1, True)} unidades</b> apostando 1 a cada partido. Apostar siempre al favorito de Bet365 da {_b["ok_mk"]} aciertos y {es(_b["unidades_mk"], 1, True)} unidades.</p>
        <div class="mini-campo"><div class="leyenda"><span><i></i>Modelo</span><span><i class="casa"></i>Favorito de Bet365</span></div>{_curva}</div>
      </div>
    </figure>
  </div>
</div></section>
'''

CIFRAS = dict(
    primera=temps[0]["t"], ultima=ult["t"], n=es(T["n"]), temporadas=len(temps), temporadas_letra=LETRAS.get(len(temps), str(len(temps))),
    acierto=es(T["acierto"], 1) + " %", acierto_mk=es(T["acierto_mk"], 1) + " %", roi=es(T["roi"], 1, True) + " %", roi_mk=es(T["roi_mk"], 1, True) + " %",
    unidades=es(T["unidades"], 1, True), unidades_mk=es(T["unidades_mk"], 1, True), ganadas=sum(t["ventaja"] > 0 for t in temps),
    ventaja=es(T["ventaja"], 2, True), lo=es(T["lo"], 2, True), hi=es(T["hi"], 2, True), jornada_actual=jor_act,
    pct_dis=es(T["pct_dis"], 1) + " %", n_dis=es(X["n"]), dis_ac=es(X["acierto"], 1) + " %", dis_ac_mk=es(X["acierto_mk"], 1) + " %",
    dis_u=es(X["unidades"], 1, True), dis_u_mk=es(X["unidades_mk"], 1, True), ll=es(T["ll"], 3), ll_mk=es(T["ll_mk"], 3),
    pct_empates=es(100 * (jug["y_true"] == "D").mean()) + " %", n_pred_empate=es((jug["y_pred"] == "D").sum()), ac_empate=es(100 * jug.loc[jug["y_pred"] == "D", "correct"].mean(), 1) + " %",
    pct_pred_empate=es(100 * (jug["y_pred"] == "D").mean(), 1) + " %",
    c_partidos=es(K["partidos"]), c_desde=K["desde"], c_stats=es(K["stats"]), c_tipos=es(K["tipos"]), c_vars=es(K["variables"]), c_entren=es(K["entrenables"]),
    n_vars=len(MODELO["features"]) if MODELO else "", ventana=es(W_), recientes=es(MODELO["season_size"]) if MODELO else "",
    peso_letra={2.0: "doble", 3.0: "triple", 4.0: "cuádruple"}.get(float(MODELO["recent_weight"]), es(MODELO["recent_weight"], 1) + " veces más") if MODELO else "",
    c_fechas=es(K["fechas"]), momento=f"de antes de la jornada {proxima[0]['j']}" if proxima else "de hoy", margen=es(T["margen"], 1) + " %",
    titulo_jornada=tit_j, texto_jornada=txt_j, seccion_chat=seccion_chat, nav_chat=nav_chat, autor=AUTOR, enlaces=f'<div class="enlaces">{enl}</div>' if enl else "", hasta=dia(DATOS["hasta"]), generado=dia(DATOS["generado"]))

cuerpo = re.sub(r"\{\{(\w+)\}\}", lambda m: str(CIFRAS[m.group(1)]), CUERPO)
datos_js = json.dumps(DATOS, ensure_ascii=False, separators=(",", ":"), allow_nan=False).replace("</", "<\\/")
icono = "data:image/svg+xml," + "%3Csvg xmlns='http://www.w3.org/2000/svg' viewBox='0 0 32 32'%3E%3Crect width='32' height='32' rx='7' fill='%230e4a37'/%3E%3Ctext x='16' y='23' font-family='Arial' font-weight='800' font-size='19' text-anchor='middle' fill='%23ffd23f'%3E1%3C/text%3E%3C/svg%3E"
descr = f"Modelo que pronostica el 1X2 de LaLiga y se compara con Bet365 en {es(T['n'])} partidos: {CIFRAS['acierto']} de acierto frente a {CIFRAS['acierto_mk']}."
redes = (f'<meta property="og:url" content="{DIRECCION}"><meta property="og:image" content="{DIRECCION}portada.png">'
         f'<meta name="twitter:card" content="summary_large_image">') if DIRECCION and (WEB / "portada.png").exists() else ""
PAGINA = f"""<!doctype html>
<html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">
<title>Pronóstico LaLiga</title><meta name="description" content="{descr}"><meta property="og:title" content="Pronóstico LaLiga"><meta property="og:description" content="{descr}"><meta property="og:type" content="website">{redes}<meta name="theme-color" content="#0a3a2b">
<link rel="icon" href="{icono}"><link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Archivo:wdth,wght@62..125,400..900&amp;family=Hanken+Grotesk:wght@400..700&amp;display=swap">
<style>
{ESTILO}</style></head><body>
{cuerpo}
<script type="application/json" id="datos">{datos_js}</script>
<script>
{CODIGO}</script></body></html>
"""

# ── control: lo que enseña la página es lo que publicó MODELADO ──
fallos = []
for t in temps:
    for k, c, m in (("acierto", "hit_rate", met), ("roi", "roi", met), ("acierto_mk", "hit_rate", metk), ("roi_mk", "roi", metk)):
        if abs(t[k] / 100 - m.loc[t["s"], c]) > 1e-9:
            fallos.append(f"{t['t']}: {k} no coincide con metrics_*")
if abs(sum(t["unidades"] for t in temps) - T["unidades"]) > 1e-6 or abs(DATOS["curva"][-1][3] - T["unidades"]) > .011:
    fallos.append("las unidades por temporada no suman el total")
if re.search(r"\{\{|\bnan\b", cuerpo):
    fallos.append("queda un hueco sin rellenar en los textos")
for x in proxima:
    if abs(sum(x["p"]) - 1) > 1e-3 or x["y"] != int(np.argmax(x["p"])):
        fallos.append(f"pronóstico incoherente: {x['l']}-{x['v']}")
sin_escudo = [e["n"] for e in DATOS["equipos"] if not e["e"]]
if fallos:
    raise ValueError("no se escribe la página:\n   " + "\n   ".join(sorted(set(fallos))[:10]))
WEB.mkdir(exist_ok=True)
SALIDA = WEB / "index.html"
SALIDA.write_text(PAGINA, encoding="utf-8", newline="\n")
print(f"✔ {SALIDA.relative_to(ROOT)} · {len(PAGINA.encode('utf-8')) / 1024:,.0f} KB · {T['n']:,} partidos · {len(proxima)} por jugar")
print(f"Coincide con las métricas de MODELADO en las {len(temps)} temporadas." + (f" Sin escudo (van con iniciales): {', '.join(sin_escudo)}." if sin_escudo else ""))
if LM:
    print(f"Registro limpio desde la jornada {LIMPIO_DESDE[1]} de {temporada(LIMPIO_DESDE[0])} (no sale en la página): "
          f"modelo {LM['ok']}/{LM['n']} y {es(LM['unidades'], 2, True)} u · Bet365 {LM['ok_mk']}/{LM['n']} y {es(LM['unidades_mk'], 2, True)} u")
if ABRIR:
    webbrowser.open(SALIDA.as_uri())

✔ docs\index.html · 509 KB · 4,689 partidos · 10 por jugar
Coincide con las métricas de MODELADO en las 13 temporadas.
